## 0 · The Challenge

> **The mission:** Riverside House must adapt a hardware-appropriate local SmolLM2 checkpoint without sending its confidential manuscript corpus to a public API.

**What we know so far:**

- The CPU profile is small enough to expose every mechanism; CUDA profiles trade more memory for stronger starting behavior.
- Riverside can test every candidate with the same three acceptance probes: catalog fluency, instruction compliance, and editor preference.
- **But the starting model fails the first probe:** it has never seen Riverside's unpublished stories.

**What's blocking us:**
The model has never experienced Riverside's corpus. Giving it the manuscripts may improve continuation, but that only practices prose. If it later ignores a bounded editor request, that failure must determine the next training experience. If it follows the request but chooses a needlessly verbose answer, that remaining failure determines the final one.

**What this chapter unlocks:**
A failure-driven training path. We expose one failed probe, apply the smallest data-objective change that addresses it, then rerun the probes before adding another technique. Weak 135M outputs remain acceptable when the code and evidence path are correct and the notebook labels the capacity limit honestly.

# LLM Fine-Tuning Deep Dive, Part 1 of 3: What Should the Model Learn?

> **The story:** Riverside changes the model's training experience only when an observed failure justifies it: unfamiliar Aria prose motivates continued pretraining, ignored requests motivate SFT, and inferior choices among valid answers motivate preference learning.
>
> **Where you are:** The transformer chapters explained next-token prediction. Part 1 changes **what behavior the examples teach**; Part 2 changes **where the update is stored**; Part 3 asks **what the evidence supports**.
>
> **Terms:** a preference example contains one prompt, a response an editor would keep, and a response the editor would reject. Concept 3 derives how those comparisons can become a training signal.

## The Aria Test Story

> **Fictional corpus context:** This notebook uses only Riverside's original 40-chapter generation-ship novel *The Weight of Distant Light*. Aria Voss is a Systems Maintenance technician aboard the *Meridian's Promise* whose discovery of an artificial prime-number signal begins the novel's first-contact arc with the ancient distributed intelligence later known as the Choir. Chapters 1-36 provide training material; chapters 37-40 remain untouched evaluation data.

Aria begins by diagnosing the ship's aging systems with her mentor Petrus Okonkwo and the Keeper AI Wren. The signal leads her from routine maintenance into the Lantern contact team, where she helps interpret the warning carried by the transmission while remaining connected to Nyla Kade and the Under-Hold community.

## Riverside's Brief

Riverside House is adapting *The Weight of Distant Light* without sending manuscript text to a public API. The notebook pins a matched SmolLM2 base/instruction pair selected from the available CPU or CUDA memory.

The immediate job is an editing assistant that can continue Aria's story, obey a bounded request, and favor the response an editor would keep. Those are distinct behaviors, so each receives its own training signal and acceptance probe.

| Part | Question |
| --- | --- |
| 1 - this notebook | What training experience addresses the observed failure? |
| [2 - parameter strategy](02-llm-finetuning-parameter-techniques.ipynb) | How much model state must move, and what remains resident? |
| [3 - comparison and decision](03-llm-finetuning-comparison-and-decision.ipynb) | What does each result prove, and what can ship? |

The runnable checkpoints are not one mandatory ancestry chain: continued pretraining and SFT start from separate base models; the preference stage continues from the accepted SFT adapter.

## Corpus and Setup

This notebook reserves the final four Aria chapters before any examples are built. The first 36 chapters may supply training data; the reserved chapters supply evaluation only. Checkpoints and provenance manifests are written under `checkpoints/llm-finetuning/<profile>/`, shared by Parts 2 and 3.

> **Boundary:** fine-tuning changes persistent behavior. A later retrieval chapter supplies current, citable manuscript facts.

## Fine-Tuning Roadmap: Start Here

This map stays with the three-notebook arc. Read it vertically: Part 1 teaches **what behavior to learn**, Part 2 changes **how many parameters learn it**, and Part 3 decides **which evidence matters for each workload**.

```mermaid
flowchart TD
    Start["Starting point<br/>Pinned base SmolLM2: fluent, but Aria-blind"]

    subgraph Data["Part 1 - Current notebook: choose the learning objective"]
        direction TB
        C1["[Now] Concept 1<br/>Continued pretraining<br/>Make reserved Aria prose less surprising"]
        C2["[Next] Concept 2<br/>SFT<br/>Specialize a one-sentence contract"]
        C3["[Next] Concept 3<br/>Preference alignment<br/>Rank valid answers using comparisons"]
        C1 --> C2 --> C3
    end

    subgraph Params["Part 2 - Next notebook: choose the parameter strategy"]
        direction TB
        C4["Concept 4<br/>Full fine-tuning"]
        C5["Concept 5<br/>Partial freezing"]
        C6["Concept 6<br/>LoRA"]
        C7["Concept 7<br/>QLoRA + quantization"]
        C4 --> C5 --> C6 --> C7
    end

    Start --> C1
    C3 --> Saved["Part 1 checkpoint<br/>Three objective-specific artifacts and manifests"]
    Saved --> C4
    C7 --> Compare["Part 3<br/>Compare evidence and choose per workload"]
```

> The arrows are a learning sequence, not literal model ancestry. Continued pretraining and SFT start from separate pinned base models; the preference stage continues from the accepted SFT adapter. Every objective is judged on the four Aria chapters reserved before training examples are built.

## Prerequisite Bridge: From Encoder-Decoder Attention to a Decoder-Only Assistant

The transformer foundations introduced three useful shapes: an **encoder** reads an entire input, a **decoder** predicts the next token while respecting a causal mask, and an **encoder-decoder** model lets a decoder attend to an encoded source through cross-attention. Riverside's assistant uses the decoder-only choice: at each turn, the user's instruction, any supplied scene, and the completion form one growing token sequence.

| Foundation | Role in this chapter | Why Riverside needs it |
| --- | --- | --- |
| Causal decoder | The selected SmolLM2 profile predicts the next token | It can continue prose and answer prompts from one left-to-right context |
| Training objective | Labels say which next tokens should become more likely | Continued pretraining, SFT, and preference learning change what Riverside teaches the decoder |
| Encoder / retrieval later | Encodes a query and passages for matching | It finds current, citable evidence instead of asking the generator to remember every fact |

So this notebook changes **how a decoder-only model behaves**. It does not turn the model into a dependable catalog lookup system.

The underlying mechanics are owned by the prerequisite notebooks:

- [Transformers Part 12](../02-transformers/02-decoder-only-language-model.ipynb#part-12---the-causal-triangle-and-the-accumulation-tower) explains causal visibility.
- [Transformers Part 8](../02-transformers/02-decoder-only-language-model.ipynb#part-8---mini-language-model-training--inference) owns decoder-only training, including the [per-position loss microscope](../02-transformers/02-decoder-only-language-model.ipynb#per-position-loss-one-sequence-many-lessons) and [one complete backward/update trace](../02-transformers/02-decoder-only-language-model.ipynb#one-backward-pass-many-token-lessons-one-update).
- [PyTorch RNN Bridge Part 4](../01-rnns/01-pytorch-rnn-bridge.ipynb#part-4--explicit-sequence-training-and-gradient-clipping) owns shifted sequence loss, backpropagation, and optimizer mechanics.
- [Encoder-Decoder Part 5](../02-transformers/03-encoder-decoder-and-cross-attention.ipynb#part-5--full-encoder-decoder-training) owns teacher-forced seq2seq training and cross-attention.

This notebook assumes those mechanics and focuses on the fine-tuning decision: **which examples and labels teach the behavior Riverside needs?**

> **Implementation preview:** the objective and parameter strategy are separate choices. This notebook uses full fine-tuning for the continued-pretraining demonstration, then small LoRA adapters for SFT and the final preference update so the runs fit local hardware. Treat LoRA here as a small trainable correction attached to a frozen base; Part 2 opens that black box and compares it with full and partial fine-tuning.

## Learning Route

1. Pin the base revision, seed, artifact paths, and Aria chapter split.
2. Let catalog-fluency failure create the need for continued pretraining.
3. Decide continued pretraining from reserved-chapter perplexity, not training loss.
4. Let the one-sentence contract create the need for response-masked SFT.
5. Decide SFT from complete passes on unseen wording and reserved contexts.
6. Let valid-but-inferior answers create the need for preference evidence.
7. Derive reward-model optimization and PPO, then simplify the fixed-pair path into DPO.
8. Continue to Part 2 for parameter cost and Part 3 for broader workload evidence.

**Optional depth:** token-level mechanics live in the prerequisite notebooks linked below. The resumable-job appendix remains a reference; it does not replace the explicit evaluation gates.

In [ ]:
from pathlib import Path

# Resolve the notebook directory once so corpus and checkpoint paths do not depend on kernel cwd.
try:
    _notebook_dir = Path(__vsc_ipynb_file__).parent  # type: ignore[name-defined]
except NameError:
    try:
        _notebook_dir = Path(__file__).parent
    except NameError:
        _notebook_dir = Path.cwd()

REPO_ROOT = _notebook_dir.parents[2]
CONTENT_DIR = _notebook_dir / "content"
CHECKPOINT_DIR = REPO_ROOT / "checkpoints"
ARIA_NOVEL_DIR = CONTENT_DIR / "the-weight-of-distant-light"

if not ARIA_NOVEL_DIR.exists():
    raise FileNotFoundError(
        f"Could not find the Aria corpus at {ARIA_NOVEL_DIR}. Open and run this notebook from "
        "learning/genai/03-llm-finetuning so the committed content directory resolves correctly."
    )

ARIA_CHAPTER_FILES = sorted(ARIA_NOVEL_DIR.glob("chapter-*.txt"))
ARIA_HOLDOUT_CHAPTER_COUNT = 4
if len(ARIA_CHAPTER_FILES) <= ARIA_HOLDOUT_CHAPTER_COUNT:
    raise ValueError("The Aria corpus needs more chapters than the configured holdout count")

ARIA_TRAIN_FILES = ARIA_CHAPTER_FILES[:-ARIA_HOLDOUT_CHAPTER_COUNT]
ARIA_HOLDOUT_FILES = ARIA_CHAPTER_FILES[-ARIA_HOLDOUT_CHAPTER_COUNT:]
assert set(ARIA_TRAIN_FILES).isdisjoint(ARIA_HOLDOUT_FILES)


def load_paragraphs(chapter_files, min_len=200):
    """Load qualifying paragraphs from an explicit, provenance-preserving chapter split."""
    paragraphs = []
    for path in chapter_files:
        text = path.read_text(encoding="utf-8")
        for paragraph in text.split("\n\n"):
            paragraph = paragraph.strip().replace("\n", " ")
            if len(paragraph) >= min_len:
                paragraphs.append(paragraph)
    return paragraphs


sample_paragraphs = load_paragraphs(ARIA_TRAIN_FILES[:2])
print(f"Aria corpus: {len(ARIA_CHAPTER_FILES)} chapters")
print(f"Training split: {len(ARIA_TRAIN_FILES)} chapters")
print(f"Held-out split: {len(ARIA_HOLDOUT_FILES)} chapters")
print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"First training paragraph:\n\n{sample_paragraphs[0][:421]} ...")

Aria corpus: 40 chapters
Training split: 36 chapters
Held-out split: 4 chapters
Checkpoints: c:\r\ai-portfolio\checkpoints
First training paragraph:

Aria Voss had learned to listen to the ship the way other people listened to weather. Two hundred and fourteen years after the Meridian's Promise had folded its sails of solar canvas and slipped away from the blue marble of Earth, the great vessel still spoke in a thousand small voices, and Aria had spent her whole life learning to tell them apart. The groan of the spinward bearings when the ring decks turned through  ...


## Baseline: Let the Model Fail Before Naming a Technique

Riverside will reuse three probes after every training stage:

| Probe | Request | Failure signal |
| --- | --- | --- |
| Catalog fluency | Continue a passage containing Riverside-only names and relationships | Generic continuation or invented story facts |
| Instruction compliance | `Answer in one sentence and stop.` | Restates the request, rambles, or violates the format |
| Editor preference | Compare two valid answers to the same request | No consistent reason to favor the concise, useful answer |

Start with the catalog-fluency probe. The base model has never seen Riverside's manuscripts, so a fluent answer is not evidence of knowledge. It can only guess from names in the prompt.

That gives us the first failure to fix:

> The model knows how English works, but Riverside language is still surprising to it.

---

### Setting Up the Shared Baseline

All three stages use the same base checkpoint and fixed prompts. Keeping an untouched `base_model` gives every later comparison a real before state rather than a remembered sample.

### Pick Capacity From the Hardware, Keep the Objective Fixed

The notebook chooses a matched base/instruction pair from one model family:

| Runtime | Continued-pretraining base | SFT/DPO base | Purpose |
| --- | --- | --- | --- |
| CPU | `HuggingFaceTB/SmolLM2-135M` | `HuggingFaceTB/SmolLM2-135M-Instruct` | Keep every mechanism runnable on ordinary hardware. |
| CUDA below 64 GiB | `HuggingFaceTB/SmolLM2-360M` | `HuggingFaceTB/SmolLM2-360M-Instruct` | Give a typical GPU a stronger baseline without making the multi-model comparison impractical. |
| CUDA with at least 64 GiB | `HuggingFaceTB/SmolLM2-1.7B` | `HuggingFaceTB/SmolLM2-1.7B-Instruct` | Prefer the strongest same-family teaching checkpoint when memory supports the complete arc. |

The objective, prompt contract, LoRA targets, data split, and evidence gates do not change between profiles. Artifacts are written to a profile-specific directory because adapters and full checkpoints cannot be moved safely between model sizes.

In [ ]:
import hashlib
import json
import random
from importlib.metadata import version

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

CUDA_AVAILABLE = torch.cuda.is_available()
GPU_MEMORY_GIB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3 if CUDA_AVAILABLE else 0.0
)

if not CUDA_AVAILABLE:
    MODEL_PROFILE = "cpu-small-135m"
    CPT_MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
    CPT_MODEL_REVISION = "93efa2f097d58c2a74874c7e644dbc9b0cee75a2"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
    INSTRUCT_MODEL_REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"
elif GPU_MEMORY_GIB < 64:
    MODEL_PROFILE = "gpu-balanced-360m"
    CPT_MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
    CPT_MODEL_REVISION = "f8027fd0eaeea54caa13c31d31b9fdc459c38b49"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"
    INSTRUCT_MODEL_REVISION = "a10cc1512eabd3dde888204e902eca88bddb4951"
else:
    MODEL_PROFILE = "gpu-quality-1.7b"
    CPT_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B"
    CPT_MODEL_REVISION = "effd688a12921b4cc83e3312b6feb579f70f9c71"
    INSTRUCT_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
    INSTRUCT_MODEL_REVISION = "31b70e2e869a7173562077fd711b654946d38674"

MODEL_NAME = INSTRUCT_MODEL_NAME
MODEL_REVISION = INSTRUCT_MODEL_REVISION
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
EXPERIMENT_SEED = 2026
DEMO_TRAIN_STEPS = 10
DEMO_DPO_STEPS = 10
CPU_BUDGET_MINUTES = 360
GPU_BUDGET_MINUTES = 20

PROFILE_CHECKPOINT_DIR = CHECKPOINT_DIR / "llm-finetuning" / MODEL_PROFILE
CPT_CHECKPOINT_DIR = PROFILE_CHECKPOINT_DIR / "non-instruction-full"
SFT_CHECKPOINT_DIR = PROFILE_CHECKPOINT_DIR / "instruction-lora"
DPO_CHECKPOINT_DIR = PROFILE_CHECKPOINT_DIR / "preference-dpo"

random.seed(EXPERIMENT_SEED)
set_seed(EXPERIMENT_SEED)


def _file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_artifact_manifest(
    stage,
    output_dir,
    training_files,
    evaluation_files,
    training_args,
    extra=None,
    model_name=MODEL_NAME,
    model_revision=MODEL_REVISION,
):
    """Record immutable model, data-split, package, and training provenance beside an artifact."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    def describe(files):
        return [
            {"path": path.relative_to(REPO_ROOT).as_posix(), "sha256": _file_sha256(path)}
            for path in files
        ]

    manifest = {
        "stage": stage,
        "model_profile": MODEL_PROFILE,
        "model": {"id": model_name, "revision": model_revision},
        "seed": EXPERIMENT_SEED,
        "training_files": describe(training_files),
        "evaluation_files": describe(evaluation_files),
        "training_arguments": training_args.to_dict(),
        "packages": {
            package: version(package)
            for package in ("torch", "transformers", "datasets", "peft", "trl")
        },
        "extra": extra or {},
    }
    manifest_path = output_dir / "experiment-manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    print(f"Wrote provenance manifest: {manifest_path}")
    return manifest_path


training_budget_minutes = GPU_BUDGET_MINUTES if CUDA_AVAILABLE else CPU_BUDGET_MINUTES
print(f"Model profile: {MODEL_PROFILE}")
print(f"CUDA memory: {GPU_MEMORY_GIB:.1f} GiB")
print(f"CPT model: {CPT_MODEL_NAME}@{CPT_MODEL_REVISION[:8]}")
print(f"SFT/DPO model: {INSTRUCT_MODEL_NAME}@{INSTRUCT_MODEL_REVISION[:8]}")
print(f"Profile checkpoints: {PROFILE_CHECKPOINT_DIR}")
print(f"Training budget on this device: {training_budget_minutes} minutes")
if not CUDA_AVAILABLE:
    print(
        "CPU disclaimer: the 135M model may miss known facts, ignore exact response contracts, "
        "or show little visible change after ten updates. Its limited capacity, weak instruction "
        "prior, narrow teaching corpus, and short optimization budget make that expected; use the "
        "cells to learn the objective and evidence path, not to promise production-quality prose."
    )

Model profile: cpu-small-135m
CUDA memory: 0.0 GiB
CPT model: HuggingFaceTB/SmolLM2-135M@93efa2f0
SFT/DPO model: HuggingFaceTB/SmolLM2-135M-Instruct@12fd25f7
Profile checkpoints: c:\r\ai-portfolio\checkpoints\llm-finetuning\cpu-small-135m
Training budget on this device: 360 minutes
CPU disclaimer: the 135M model may miss known facts, ignore exact response contracts, or show little visible change after ten updates. Its limited capacity, weak instruction prior, narrow teaching corpus, and short optimization budget make that expected; use the cells to learn the objective and evidence path, not to promise production-quality prose.


### Read Small-Model Results Honestly

The CPU path is intentionally a mechanism-sized experiment, not a quality promise. A 135M model can demonstrate real gradient updates, masking, checkpointing, and held-out measurement while still producing weak prose or no visible behavioral change.

Expected reasons include:

- **Limited capacity:** 135M parameters leave much less room for story knowledge and instruction behavior than the GPU profiles.
- **Weak starting behavior:** the base checkpoint is not instruction-tuned, and the tiny instruction checkpoint is less reliable than larger assistants.
- **Narrow evidence:** one novel and a small reserved suite cannot establish broad capability.
- **Short optimization:** ten updates are enough to prove that training ran, not enough to guarantee convergence.
- **Decoding sensitivity:** a probability improvement may not change greedy or sampled text.

Therefore `PASS`, `FAIL`, and `INCONCLUSIVE` remain meaningful. An inconclusive CPU output is expected evidence about the model-and-budget combination, not a failure of CPT, SFT, DPO, or LoRA as concepts. CUDA automatically selects a stronger same-family model, but its outputs still have to clear the same reserved gates.

> **PyTorch → Keras:** `torch.cuda.is_available()` — checks whether a CUDA-capable GPU is visible to PyTorch and returns a bool; the result picks the `device` string (`"cuda"` or `"cpu"`) that every tensor and model call below is pinned to via `.to(device)`. **Keras/TF equivalent:** `tf.config.list_physical_devices('GPU')` — TensorFlow auto-places ops on any visible GPU without needing an explicit device string threaded through the code, so most Keras code skips this check entirely; `tf.device(...)` exists for the rare case you want to force placement.

In [ ]:
device = "cuda" if CUDA_AVAILABLE else "cpu"
print(f"Using device: {device} ({MODEL_PROFILE})")

Using device: cpu (cpu-small-135m)


### Loading the Tokenizers

Continued pretraining and instruction-tuned generation use different checkpoints, so each path loads the tokenizer pinned to its own model revision. The selected SmolLM2 checkpoints share a 49,152-token vocabulary, and their EOS token also serves as the padding token.

This section stays at the token level: how text becomes IDs, why whitespace changes token boundaries, and how padding is excluded from loss. Role-based instruction prompts and response-only supervision begin in the SFT section.

> **PyTorch → Keras:** `AutoTokenizer.from_pretrained(...)` loads the same checkpoint-matched tokenizer for either framework. Tokenization is framework-agnostic; the split between PyTorch and TensorFlow begins when the resulting arrays become framework tensors.

In [ ]:
# Load the tokenizer pinned to each model path before comparing their token contracts.
cpt_tokenizer = AutoTokenizer.from_pretrained(
    CPT_MODEL_NAME,
    revision=CPT_MODEL_REVISION,
)
if cpt_tokenizer.pad_token is None:
    cpt_tokenizer.pad_token = cpt_tokenizer.eos_token

tokenizer = AutoTokenizer.from_pretrained(
    INSTRUCT_MODEL_NAME,
    revision=INSTRUCT_MODEL_REVISION,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"CPT tokenizer vocabulary: {len(cpt_tokenizer):,}")
print(f"CPT EOS/pad: {cpt_tokenizer.eos_token!r} / {cpt_tokenizer.pad_token!r}")
print(f"Instruction-checkpoint tokenizer vocabulary: {len(tokenizer):,}")
print(f"Instruction-checkpoint EOS/pad: {tokenizer.eos_token!r} / {tokenizer.pad_token!r}")

### Seeing the Vocabulary in Action

Byte-level BPE can represent arbitrary text, but common spans receive compact tokens while rare names or
unusual Unicode sequences split into several pieces. Whitespace is context: tokenizing `"signal"` and
`" signal"` can produce different IDs because a space may be merged with neighboring bytes.

The code below prints IDs, raw tokenizer tokens, and decoded pieces for each example. Decoding each ID is
the portable way to make spaces and newlines visible; raw token strings are implementation details and
should not be treated as a universal notation.


> **PyTorch → Keras:** `tokenizer.encode(word)` / `tokenizer.convert_ids_to_tokens(ids)` — converts raw text to integer token IDs (and back to readable BPE-piece strings) using the framework-agnostic tokenizer loaded above; no tensors are created yet, just plain Python lists. **Keras/TF equivalent:** identical call — `AutoTokenizer` isn't PyTorch- or TF-specific, so a Keras/TF version of this notebook would use this exact same code; only the downstream model call (`TFAutoModelForCausalLM` vs. `AutoModelForCausalLM`) would differ.

In [ ]:
# One example each of a noun, proper noun, verb, and adjective -- all pulled from Riverside's own
# sci-fi opening line, so these are words this notebook already leans on elsewhere.
example_words = {
    "noun": "signal",
    "proper noun": "Aria",
    "verb": "stared",
    "adjective": "distant",
}

for part_of_speech, word in example_words.items():
    ids_alone = tokenizer.encode(word)  # tokenize the word standalone (no leading space)
    ids_mid_sentence = tokenizer.encode(" " + word)  # tokenize as it would appear mid-sentence
    print(f"{part_of_speech.upper()}: {word!r}")
    print(
        f"  as the first word of a text  : ids={ids_alone}  "
        f"tokens={tokenizer.convert_ids_to_tokens(ids_alone)}"
    )
    print(
        f"  mid-sentence (' {word}')".ljust(31) + f": ids={ids_mid_sentence}  "
        f"tokens={tokenizer.convert_ids_to_tokens(ids_mid_sentence)}"
    )
    print()

print(
    "'\u0120' at the start of a token marks a leading space -- it's why the same word can tokenize "
    "differently depending on where it appears in a sentence."
)


NOUN: 'signal'
  as the first word of a text  : ids=[21164]  tokens=['signal']
  mid-sentence (' signal')     : ids=[4973]  tokens=['Ġsignal']

PROPER NOUN: 'Aria'
  as the first word of a text  : ids=[49, 5484]  tokens=['A', 'ria']
  mid-sentence (' Aria')       : ids=[330, 5484]  tokens=['ĠA', 'ria']

VERB: 'stared'
  as the first word of a text  : ids=[302, 1214]  tokens=['st', 'ared']
  mid-sentence (' stared')     : ids=[328, 1214]  tokens=['Ġst', 'ared']

ADJECTIVE: 'distant'
  as the first word of a text  : ids=[8184, 403]  tokens=['dist', 'ant']
  mid-sentence (' distant')    : ids=[9931]  tokens=['Ġdistant']

'Ġ' at the start of a token marks a leading space -- it's why the same word can tokenize differently depending on where it appears in a sentence.


> **You may wonder:** since `Aria` splits into two tokens and appears constantly in this corpus, why
> not just train the tokenizer on Riverside's own text and merge it into a dedicated token? Two
> reasons this is out of scope for fine-tuning: extending the vocabulary adds a new, untrained row
> to the embedding matrix and output head, and filling that row in with a meaningful representation
> is itself a training problem, not something fine-tuning does for free. And splitting `Aria` into
> two tokens doesn't stop the model from learning what it means -- it can still learn to associate
> that two-token pattern with everything fine-tuning teaches it about her; it just costs two sequence
> positions instead of one, a small efficiency tax, not a correctness problem. The actual gap the
> rest of this notebook closes is that the model has never seen who Aria Voss is, not how her name
> happens to be tokenized.



> **A related question:** what happens with a word the tokenizer has rarely encountered? Byte-level
> BPE does not need an unknown-word vocabulary entry: when no longer merge matches, it falls back to smaller
> byte-derived pieces. An uncommon name such as `Itzpapalotl` therefore remains representable, although it
> usually consumes more tokens than a frequent word. Fine-tuning can improve how the model uses that sequence,
> but it does not add a new vocabulary row unless the tokenizer and embedding matrix are explicitly resized.


### Loading the Base Model

`base_model` is the actual pretrained neural network -- a checkpoint-defined number of real weights downloaded from the
Hugging Face hub, moved onto whichever device we resolved above via `.to(device)`. This untouched
checkpoint is the "before" every fine-tuning technique in this notebook is compared against.


> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)` — downloads the weights selected by `MODEL_NAME` into a PyTorch `nn.Module` and moves every parameter tensor onto `device` (CPU or GPU) in place. **Keras/TF equivalent:** `TFAutoModelForCausalLM.from_pretrained(MODEL_NAME)` — loads the same checkpoint into a `tf.keras.Model` instead; TensorFlow doesn't need an explicit `.to(device)` call since ops are placed on available devices automatically (or via a `tf.device(...)` context).

In [ ]:
# Load the untouched instruction baseline used by SFT, DPO, and general-knowledge probes.
base_model = AutoModelForCausalLM.from_pretrained(
    INSTRUCT_MODEL_NAME,
    revision=INSTRUCT_MODEL_REVISION,
).to(device)
model_parameter_count = sum(parameter.numel() for parameter in base_model.parameters())
decoder_blocks = base_model.model.layers
n_blocks = len(decoder_blocks)
hidden_size = base_model.config.hidden_size
print(
    f"Loaded {INSTRUCT_MODEL_NAME}@{INSTRUCT_MODEL_REVISION[:8]}: "
    f"{model_parameter_count:,} parameters, {n_blocks} decoder blocks, "
    f"hidden size {hidden_size}."
)

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 2522.63it/s]

Loaded HuggingFaceTB/SmolLM2-135M-Instruct@12fd25f7: 134,515,008 parameters, 30 decoder blocks, hidden size 576.


### A Fixed Test Prompt for Before/After Comparisons

`PROMPT` is the one fixed test sentence reused throughout the notebook so "before" vs. "after"
fine-tuning comparisons are always apples-to-apples. It's pulled straight from the sci-fi corpus so
a model that has actually absorbed the catalog has a real chance of continuing it in-world.


In [ ]:
PROMPT = "Aria Voss stared at the signal counting itself out in prime numbers and"  # from the sci-fi corpus

### A Reusable `generate()` Helper

Hugging Face returns the prompt and completion in one token sequence. The helper records the prompt length, slices `output[prompt_length:]`, and decodes only the new tokens so every comparison shows the model's actual continuation.

It also calls `.strip()` because the first generated token may carry leading whitespace. All later candidates use this same helper, so output formatting cannot masquerade as a model difference.

> **PyTorch → Keras:** `model.eval()` / `torch.no_grad()` / `model.generate()` — `.eval()` switches dropout/batchnorm-style layers to inference mode, `torch.no_grad()` disables gradient tracking to save memory during inference, and `.generate()` runs HuggingFace's autoregressive sampling loop (nucleus sampling here via `top_p`/`temperature`). **Keras/TF equivalent:** `TFAutoModelForCausalLM.generate()` — the same HuggingFace `.generate()` API exists on TF models with identical sampling arguments; TF's analog of "eval mode" is passing `training=False` (implicit inside `.generate()`), and there's no separate "no_grad" context since calling a `tf.keras.Model` outside a `GradientTape` block already skips gradient recording.

In [ ]:
def generate(model, prompt, max_new_tokens=60):
    """Generate a text continuation for *prompt*.

    Returns **only the newly generated tokens** (prompt is stripped), so every
    print(generate(...)) call in this notebook shows the model's actual output
    without echoing the input back.

    Parameters
    ----------
    model : PreTrainedModel or PeftModel
        Any HuggingFace causal-LM model (base model, LoRA adapter, DPO policy …)
    prompt : str
        The input text passed to the model.
    max_new_tokens : int
        Hard cap on how many new tokens to generate after the prompt ends.
        The model can stop earlier if it samples the EOS token.

    Notes
    -----
    A real example from this notebook's own `PROMPT` (15 tokens) makes both
    lines concrete. Asking for `max_new_tokens=15` returns `out` with shape
    `(1, 30)` -- the 15 prompt tokens plus 15 new ones, concatenated. Decoding
    all 30 without slicing prints the prompt right back before the answer:

        'Aria Voss stared at the signal counting itself out in prime numbers
         and began to ponder the question, what was it that she had to do?'

    `out[0][prompt_len:]` (`prompt_len = 15` here) drops the first 15 tokens so
    only the new continuation gets decoded. But decoding *just* those 15 new
    tokens gives:

        ' began to ponder the question, what was it that she had to do?'

    -- note the stray leading space: the decoded continuation can begin with whitespace carried by its first
    token, so the raw decoded string may start with a space. `.strip()`
    removes it, along with any trailing whitespace/newlines near the end.
    """
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(
        device
    )  # use the same tokenizer to tokenize the prompt and convert it to tensor
    prompt_len = inputs["input_ids"].shape[1]  # track where the prompt ends
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,  # stochastic → varied output
            top_p=0.9,  # nucleus sampling: top 90% mass
            temperature=0.8,  # soften distribution slightly
            pad_token_id=tokenizer.pad_token_id,
        )

    # out[0] shape: (prompt_len + new_tokens,)
    # Slice from prompt_len onward to get ONLY the model's continuation
    completion = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    return (
        completion
        if completion
        else "[model stopped immediately — sampled EOS as first token]"
    )


print("=== Baseline (no fine-tuning) — model continuation only ===")
print(f"Prompt    : {PROMPT}")
print(f"Completion: {generate(base_model, PROMPT, 20)}")

### Code Walkthrough: Shared Tokenizer and Model Setup

**1. Native pad/EOS token**

For the selected SmolLM2 checkpoints, one special token serves as both padding and EOS. Padding positions still receive label `-100`, so they contribute no loss.

**2. Runtime-derived architecture**

`AutoModelForCausalLM.from_pretrained(...)` loads the pinned checkpoint. Parameter count, decoder-block count, and hidden width come from the loaded object rather than hard-coded prose.

## Test Prompts: Define Success Before Training

A fluent continuation is not enough; the base model is already fluent. A successful Riverside adaptation must use story-specific entities coherently without copying the prompt or inventing another world.

Use three Aria probe families:

| Probe | What it checks | Failure signal |
| --- | --- | --- |
| Character and ship | Aria's relationship to the *Meridian's Promise* | Generic roles, places, or invented lore |
| Story vocabulary | The Keeper, Lantern, signal, and Under-Hold | Names are repeated without coherent relationships |
| Reserved chapters | Improvement beyond the chapters used for training | Lower training loss without lower held-out NLL |

For example, after `Aria Voss checked the Meridian's Promise status panel and`, an adapted continuation should remain aboard Riverside's ship and use established relationships; mentioning the name alone does not count.

The executable `TEST_PROMPTS` dictionary in the next cell is an Aria-only qualitative fixture. The capability decision later uses held-out likelihood and contract metrics rather than judging these samples by eye.

In [ ]:
# Aria-only qualitative baseline. Quantitative gates use the reserved chapters later.
TEST_PROMPTS = {
    "aria_ship": "Aria Voss checked the Meridian's Promise status panel and",
    "aria_signal": "Aria Voss stared at the signal counting itself out in prime numbers and",
    "aria_keeper": "The Keeper's consciousness flickered through node seventeen as",
    "aria_under_hold": "In the Under-Hold, Aria heard the Lantern's warning and",
}


def test_corpus_knowledge(model, test_prompts=TEST_PROMPTS, max_new_tokens=50):
    """Run the fixed Aria prompts and return qualitative continuations."""
    results = {}
    prompts = {}
    for key, prompt in test_prompts.items():
        prompts[key] = prompt
        results[key] = generate(model, prompt, max_new_tokens=max_new_tokens)
    return results, prompts


print("=== BASELINE MODEL: Aria-only qualitative probes ===\n")
baseline_results, prompts = test_corpus_knowledge(base_model)
for key, output in baseline_results.items():
    print(f"[{key}]")
    print(prompts[key] + " .... " + output[:200] + "...\n")

The baseline output may reuse names from the prompt while inventing their roles or relationships. That is the failure state: fluent text is not evidence that the model expects Aria's manuscript.

The qualitative probes make the problem visible, but they do not decide whether continued pretraining worked. The quantitative decision uses the same untouched Aria chapters before and after training:

- lower held-out token NLL and perplexity indicates that reserved manuscript prose became less surprising;
- a bounded general-language regression check catches obvious forgetting;
- generated samples remain illustrations, not the acceptance metric.

If the short CPU run does not clear those gates, the notebook will report `INCONCLUSIVE` instead of inferring adaptation from training loss.

### Transformer Mechanics Live Upstream

A causal-LM training batch still follows one compact contract: token IDs enter the decoder, each position predicts the next token, padding labels use `-100`, and backpropagation updates whichever parameters remain trainable.

This chapter does not re-derive that contract. Use the prerequisite links above for causal masks, per-position cross-entropy, gradient flow, and optimizer updates. From here onward, every code path earns its place by answering a fine-tuning question:

- Which Riverside text becomes continued-pretraining data?
- Which prompt tokens must SFT hide from the loss?
- Which chosen/rejected pairs express editor preference?
- Which checkpoint and evaluation evidence supports promotion?

The next section begins at that fine-tuning-specific boundary.


In [ ]:
# Shared analysis imports used by later fine-tuning diagnostics.
import warnings

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch.nn.functional as F

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})
sns.set_theme(style="whitegrid", palette="muted")

print("Shared analysis libraries loaded.")

Shared analysis libraries loaded.


---

## The Fine-Tuning Journey: Let Each Failure Choose the Next Objective

```mermaid
flowchart TD
    A["Base model<br/>fluent, catalog-blind"] -->|"generic Riverside prose"| B["Continued pretraining"]
    B -->|"continues requests instead of obeying them"| C["SFT"]
    C -->|"valid answer, wrong editorial choice"| P["Preference comparisons"]
    P --> RM["Learn a reward model"]
    RM --> PPO["PPO on fresh policy drafts"]
    PPO -->|"fixed offline pairs; remove rollout machinery"| DPO["DPO"]
```

This is a diagnostic sequence, not mandatory checkpoint ancestry. Stop as soon as Riverside's required behavior passes its acceptance probes.

Part 1 changes the **training experience**. Part 2 separately asks whether full fine-tuning, freezing, LoRA, or QLoRA can carry that experience within Riverside's hardware and release constraints.

## Concept 1: Continued Pretraining - Make Aria Prose Less Surprising

The baseline failed on Aria-specific language because those names, relationships, and stylistic patterns were absent from its training experience.

**Minimal fix:** keep the original next-token objective, but continue training on raw paragraphs from the first 36 chapters. There are no instructions or preference labels yet; the model simply predicts the next manuscript token. This is **continued pretraining**, also called domain-adaptive pretraining.

| What this experience can teach | What it cannot teach |
| --- | --- |
| Domain vocabulary, recurring entities, prose patterns | How to obey a user request |
| Which continuations resemble Aria's manuscript | When to stop or return a required format |
| A better prior for later adaptation | Which of two acceptable answers an editor prefers |

The runnable example updates all model weights so the learning signal is easy to inspect. Part 2 will challenge that expensive parameter choice.

**Checkpoint after training:** compare token-weighted perplexity on the four untouched chapters and run the general-language retention check. A lower training loss without reserved improvement is `INCONCLUSIVE`, not evidence that continued pretraining worked.

> **Bridge to SFT:** even a successful prose update would practice manuscript continuation, not the one-sentence interaction contract. The next section targets that separate behavior.

### Code Walkthrough: `tokenize_causal()` - Preparing Text for Next-Token Prediction

This is the first point where raw Aria paragraphs become the fixed-length integer tensors a transformer consumes. `tokenize_causal()` is defined immediately before the dataset mapping that needs it. The guarded resumable-job appendix reuses the same function; SFT and DPO use response-aware contracts instead.

```python
def tokenize_causal(examples, tokenizer, max_length=64):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_overflowing_tokens=True,
    )
    tokens["labels"] = [
        [(token_id if mask == 1 else -100) for token_id, mask in zip(ids, attention)]
        for ids, attention in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    return tokens
```

**Arguments:**

| Argument | Type | Purpose |
| --- | --- | --- |
| `examples` | `dict` (HF batch) | A batch with a `"text"` column containing paragraphs from the 36 Aria training chapters. |
| `tokenizer` | `PreTrainedTokenizer` | The tokenizer loaded from the pinned model revision. |
| `max_length` | `int`, default `64` | Fixed sequence length. Long paragraphs produce overflow chunks; short final chunks are padded. |

**What it returns:** tokenizer output (`input_ids`, `attention_mask`) plus `labels`. Each real token id becomes its own causal-LM label, while every padding position becomes `-100`, PyTorch's ignore index.

`return_overflowing_tokens=True` matters: without it, truncation would discard paragraph tails. With it, the tail becomes another fixed-length training example. The final short chunk remains present with its padding labels masked.

### Optional Depth: Long Documents, Truncation, and Packing

Plain `truncation=True` is a paper cutter: without overflow handling, an 800-token example capped at 512 contributes only its first 512 tokens.

This notebook's `tokenize_causal()` also sets `return_overflowing_tokens=True`, so a long paragraph becomes multiple fixed-length chunks instead of silently losing its tail. The final short chunk is padded and its padding labels are masked.

| Training type | Common strategy | Trade-off |
| --- | --- | --- |
| SFT | Truncate or separately budget prompt and completion | Preserves pair structure, but an overlong response may still lose its tail |
| Continued pretraining | Overflow chunks or pack documents into fixed blocks | Preserves more text, but block boundaries weaken cross-boundary context |

```text
BLOCK A: [ Token 0 ... Token 127 ]
BLOCK B: [ Token 128 ... ]  <- attention starts again here
```

The first tokens in Block B cannot attend to Block A even when they continue the same paragraph. Production pipelines may use document-aware packing, block-diagonal attention, best-fit grouping, or local overlap to manage that trade-off.

For this teaching run, overflow chunks keep every paragraph tail visible while preserving a simple fixed-length loss mask.

> **PyTorch → Keras:** `from datasets import Dataset` / `from transformers import Trainer, TrainingArguments` / `dataset.map(...)` — HuggingFace's `Dataset.map()` applies `tokenize_causal()` to every example (batched, for speed), producing the `input_ids`/`attention_mask`/`labels` columns that `Trainer` (a full PyTorch training-loop wrapper: batching, forward/backward, optimizer step) consumes next. **Keras/TF equivalent:** `tf.data.Dataset.map(...)` + `model.fit(...)` — a Keras version would build a `tf.data.Dataset` pipeline with the same `.map()` call and then call the standard `model.fit(dataset, epochs=...)` in place of HuggingFace's `Trainer` (or use `TFAutoModelForCausalLM` with HuggingFace's own `Trainer`, which wraps `model.fit` under the hood).

In [ ]:
import math

from datasets import Dataset
from transformers import Trainer, TrainingArguments


def tokenize_causal(examples, tokenizer_to_use, max_length=64):
    """Create next-token labels while preserving long paragraphs as fixed-length chunks."""
    tokens = tokenizer_to_use(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_overflowing_tokens=True,
    )
    tokens["labels"] = [
        [(token_id if mask == 1 else -100) for token_id, mask in zip(ids, attention)]
        for ids, attention in zip(tokens["input_ids"], tokens["attention_mask"])
    ]
    return tokens


def evaluate_causal_nll(model, texts, tokenizer_to_use, max_length=96):
    """Return token-weighted causal NLL and perplexity for fixed evaluation text."""
    model.eval()
    total_nll = 0.0
    total_tokens = 0
    model_device = next(model.parameters()).device
    with torch.inference_mode():
        for text in texts:
            encoded = tokenizer_to_use(
                text,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = {name: tensor.to(model_device) for name, tensor in encoded.items()}
            output = model(**encoded, labels=encoded["input_ids"])
            valid_tokens = int(encoded["attention_mask"][:, 1:].sum().item())
            total_nll += float(output.loss) * valid_tokens
            total_tokens += valid_tokens
    mean_nll = total_nll / total_tokens
    return {"nll": mean_nll, "perplexity": math.exp(mean_nll), "tokens": total_tokens}


def generate_greedy_continuation(model, prompt, tokenizer_to_use, max_new_tokens=48):
    """Generate one deterministic continuation with the checkpoint-matched tokenizer."""
    encoded = tokenizer_to_use(prompt, return_tensors="pt")
    model_device = next(model.parameters()).device
    encoded = {name: tensor.to(model_device) for name, tensor in encoded.items()}
    prompt_length = encoded["input_ids"].shape[1]
    model.eval()
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer_to_use.pad_token_id,
            eos_token_id=tokenizer_to_use.eos_token_id,
        )
    return tokenizer_to_use.decode(
        generated[0][prompt_length:],
        skip_special_tokens=True,
    ).strip() or "[model stopped immediately]"


CPT_STORY_PROMPTS = (
    "Aria Voss stared at the signal counting itself out in prime numbers and",
    "The Keeper's consciousness flickered through node seventeen as",
)
CPT_KNOWLEDGE_PROMPTS = (
    "The capital of France is",
    "The capital city of France is",
    "France's capital city is",
    "Question: What is the capital of France?\nAnswer:",
)
GENERAL_RETENTION_TEXTS = [
    "Paris is the capital and largest city of France.",
    "Water freezes at zero degrees Celsius under standard pressure.",
    "The Earth orbits the Sun once each year.",
]

non_inst_paragraphs = load_paragraphs(ARIA_TRAIN_FILES)
heldout_cpt_paragraphs = load_paragraphs(ARIA_HOLDOUT_FILES)
CPT_EVAL_PARAGRAPHS = heldout_cpt_paragraphs[:32]
non_inst_dataset = Dataset.from_dict({"text": non_inst_paragraphs})
non_inst_tokenized = non_inst_dataset.map(
    lambda examples: tokenize_causal(examples, cpt_tokenizer),
    batched=True,
    remove_columns=["text"],
)

print(
    f"Continued-pretraining source: {len(non_inst_paragraphs):,} paragraphs from "
    f"all {len(ARIA_TRAIN_FILES)} Aria training chapters"
)
print(f"Tokenized training data: {len(non_inst_tokenized):,} 64-token chunks")
print(
    f"Untouched evaluation data: {len(heldout_cpt_paragraphs):,} paragraphs from "
    f"{len(ARIA_HOLDOUT_FILES)} reserved chapters"
)
assert set(ARIA_TRAIN_FILES).isdisjoint(ARIA_HOLDOUT_FILES)

Map: 100%|██████████| 653/653 [00:00<00:00, 5377.13 examples/s]

Continued-pretraining source: 653 paragraphs from all 36 Aria training chapters
Tokenized training data: 1,577 64-token chunks
Untouched evaluation data: 112 paragraphs from 4 reserved chapters


The data is ready. Load two fresh copies of the hardware-selected plain SmolLM2 checkpoint: one remains the permanent CPT baseline and one receives the ten full-fine-tuning updates.

> **PyTorch → Keras:** `AutoModelForCausalLM.from_pretrained(CPT_MODEL_NAME, revision=CPT_MODEL_REVISION).to(device)` loads an independent copy of the selected plain causal checkpoint, while `p.numel()` counts its scalar parameters. A TensorFlow version would use `TFAutoModelForCausalLM.from_pretrained(...)` and `model.count_params()`; TensorFlow places operations on available devices without PyTorch's `.to(device)` call.

In [ ]:
import gc

# Make this cell safe to rerun without retaining stale full-FT state.
for model_name in ("trainer_full", "full_ft_model", "cpt_base_model"):
    if model_name in globals():
        del globals()[model_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

cpt_base_model = AutoModelForCausalLM.from_pretrained(
    CPT_MODEL_NAME,
    revision=CPT_MODEL_REVISION,
).to(device)
full_ft_model = AutoModelForCausalLM.from_pretrained(
    CPT_MODEL_NAME,
    revision=CPT_MODEL_REVISION,
).to(device)

cpt_baseline_domain = evaluate_causal_nll(
    cpt_base_model,
    CPT_EVAL_PARAGRAPHS,
    cpt_tokenizer,
)
cpt_baseline_general = evaluate_causal_nll(
    cpt_base_model,
    GENERAL_RETENTION_TEXTS,
    cpt_tokenizer,
)
cpt_baseline_generations = {
    prompt: generate_greedy_continuation(cpt_base_model, prompt, cpt_tokenizer)
    for prompt in CPT_STORY_PROMPTS
}
CPT_KNOWLEDGE_PROMPTS = (
    "The capital of France is",
    "The capital city of France is",
    "France's capital city is",
    "Question: What is the capital of France?\nAnswer:",
)
cpt_baseline_knowledge_outputs = {
    prompt: generate_greedy_continuation(
        cpt_base_model,
        prompt,
        cpt_tokenizer,
        max_new_tokens=12,
    )
    for prompt in CPT_KNOWLEDGE_PROMPTS
}
cpt_baseline_knows_paris = any(
    "paris" in output.casefold() for output in cpt_baseline_knowledge_outputs.values()
)

print(
    f"Loaded two independent {CPT_MODEL_NAME}@{CPT_MODEL_REVISION[:8]} copies: "
    f"{sum(parameter.numel() for parameter in full_ft_model.parameters()):,} parameters each."
)
print(f"Reserved Aria baseline perplexity: {cpt_baseline_domain['perplexity']:.2f}")
print(f"General baseline perplexity: {cpt_baseline_general['perplexity']:.2f}")
print("Visible raw-model knowledge probes:")
for prompt, output in cpt_baseline_knowledge_outputs.items():
    print(f"  {prompt!r} -> {output!r}")
print(f"At least one probe contains the expected fact: {cpt_baseline_knows_paris}")

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 2968.81it/s]


Loaded two independent HuggingFaceTB/SmolLM2-135M@93efa2f0 copies: 134,515,008 parameters each.
Reserved Aria baseline perplexity: 50.54
General baseline perplexity: 19.62
Visible raw-model knowledge probes:
  'The capital of France is' -> 'the capital of the country.\n\nThe capital of France'
  'The capital city of France is' -> 'the capital of the department of Alsace. It is'
  "France's capital city is" -> 'the capital of the state of Louisiana.\n\nThe city'
  'Question: What is the capital of France?\nAnswer:' -> 'Paris.\n\nQuestion: What is the capital of France'
At least one probe contains the expected fact: True


### Configuring and Running the Trainer

`TrainingArguments` + `Trainer` is HuggingFace's standard training loop -- it handles batching, the forward/backward pass, and the optimizer step described earlier in this notebook, so we do not write that loop by hand. `DEMO_TRAIN_STEPS = 10` and `learning_rate=5e-5` keep this CPU demonstration bounded; a real Riverside training run would choose its budget from measured convergence. `trainer_full.train()` runs all ten real optimizer steps used by every later comparison in this notebook.

> **PyTorch → Keras:** `TrainingArguments(...)` / `Trainer(model=..., args=..., train_dataset=...)` / `trainer_full.train()` / `full_ft_model.save_pretrained(...)` — configures and runs HuggingFace's full PyTorch training loop (batching, forward/backward passes, optimizer steps, logging) in one `.train()` call, then serializes the fine-tuned weights + config to disk. **Keras/TF equivalent:** `model.compile(optimizer=..., loss=...)` + `model.fit(dataset, epochs=...)` — the direct Keras analog of configuring + running training; `save_pretrained(...)` has an identically-named method on `TFPreTrainedModel` subclasses, so the checkpoint-saving line itself would be unchanged in a TF version.

In [ ]:
# Ten real optimizer updates keep the mechanism runnable on every selected profile.
training_args_full = TrainingArguments(
    output_dir=str(CPT_CHECKPOINT_DIR),
    per_device_train_batch_size=1,
    max_steps=DEMO_TRAIN_STEPS,
    logging_steps=1,
    save_strategy="no",
    learning_rate=5e-5,
    seed=EXPERIMENT_SEED,
    data_seed=EXPERIMENT_SEED,
    report_to="none",
)

trainer_full = Trainer(
    model=full_ft_model,
    args=training_args_full,
    train_dataset=non_inst_tokenized,
)
trainer_full.train()
full_ft_model.save_pretrained(CPT_CHECKPOINT_DIR)
cpt_tokenizer.save_pretrained(CPT_CHECKPOINT_DIR)
write_artifact_manifest(
    stage="continued-pretraining-full",
    output_dir=CPT_CHECKPOINT_DIR,
    training_files=ARIA_TRAIN_FILES,
    evaluation_files=ARIA_HOLDOUT_FILES,
    training_args=training_args_full,
    extra={"objective": "causal language modeling", "parameter_strategy": "full"},
    model_name=CPT_MODEL_NAME,
    model_revision=CPT_MODEL_REVISION,
)
print("Saved continued-pretraining checkpoint and provenance manifest.")

### Did Continued Pretraining Make Reserved Aria Prose Less Surprising?

Training loss proves only that optimization ran. The next cell uses the final four chapters, which were excluded before training examples were built, and computes token-weighted NLL and perplexity under both the untouched base model and the trained checkpoint.

The CPU-friendly gate evaluates a fixed prefix of reserved paragraphs. It reports:

- `PASS` when held-out perplexity improves by at least 2% without more than 5% regression on a small general-language retention set;
- `FAIL` when general-language perplexity regresses by more than 5%;
- `INCONCLUSIVE` when the short run is safe but the held-out improvement is too small.

The thresholds are teaching gates, not production release criteria. Their purpose is to prevent a noisy ten-step training loss from being presented as capability evidence.

In [ ]:
trained_domain = evaluate_causal_nll(
    full_ft_model,
    CPT_EVAL_PARAGRAPHS,
    cpt_tokenizer,
)
trained_general = evaluate_causal_nll(
    full_ft_model,
    GENERAL_RETENTION_TEXTS,
    cpt_tokenizer,
)
trained_knowledge_outputs = {
    prompt: generate_greedy_continuation(
        full_ft_model,
        prompt,
        cpt_tokenizer,
        max_new_tokens=12,
    )
    for prompt in CPT_KNOWLEDGE_PROMPTS
}
trained_knows_paris = any(
    "paris" in output.casefold() for output in trained_knowledge_outputs.values()
)

base_domain = cpt_baseline_domain
base_general = cpt_baseline_general
domain_improvement_pct = 100 * (
    base_domain["perplexity"] - trained_domain["perplexity"]
) / base_domain["perplexity"]
general_regression_pct = 100 * (
    trained_general["perplexity"] - base_general["perplexity"]
) / base_general["perplexity"]
knowledge_retained = not cpt_baseline_knows_paris or trained_knows_paris

if general_regression_pct > 5.0 or not knowledge_retained:
    cpt_status = "FAIL"
elif domain_improvement_pct >= 2.0:
    cpt_status = "PASS"
else:
    cpt_status = "INCONCLUSIVE"

cpt_evidence = {
    "status": cpt_status,
    "model_profile": MODEL_PROFILE,
    "heldout_base_ppl": base_domain["perplexity"],
    "heldout_trained_ppl": trained_domain["perplexity"],
    "heldout_improvement_pct": domain_improvement_pct,
    "general_regression_pct": general_regression_pct,
    "baseline_knows_paris": cpt_baseline_knows_paris,
    "trained_knows_paris": trained_knows_paris,
    "knowledge_retained": knowledge_retained,
    "heldout_tokens": trained_domain["tokens"],
}

print("=== Continued-pretraining evidence ===")
print(
    f"Reserved Aria PPL: {base_domain['perplexity']:.2f} -> "
    f"{trained_domain['perplexity']:.2f} ({domain_improvement_pct:+.2f}% improvement)"
)
print(
    f"General-language PPL: {base_general['perplexity']:.2f} -> "
    f"{trained_general['perplexity']:.2f} ({general_regression_pct:+.2f}% regression)"
)
print(f"Visible Paris probe retained: {cpt_baseline_knows_paris} -> {trained_knows_paris}")
for prompt in CPT_KNOWLEDGE_PROMPTS:
    print(f"  Before {prompt!r} -> {cpt_baseline_knowledge_outputs[prompt]!r}")
    print(f"  After  {prompt!r} -> {trained_knowledge_outputs[prompt]!r}")
print(f"Decision: {cpt_status}")
if cpt_status == "INCONCLUSIVE" and MODEL_PROFILE == "cpu-small-135m":
    print(
        "Expected small-model limitation: ten updates may move likelihood without changing visible text. "
        "The short run and 135M capacity are intentionally insufficient for a quality guarantee."
    )

In [ ]:
# Pair the reserved likelihood metric with visible generations from the saved profile checkpoint.
comparison_base_model = AutoModelForCausalLM.from_pretrained(
    CPT_MODEL_NAME,
    revision=CPT_MODEL_REVISION,
).to(device)
comparison_cpt_model = AutoModelForCausalLM.from_pretrained(CPT_CHECKPOINT_DIR).to(device)

print("=== Base versus continued-pretraining generations ===")
for prompt in CPT_STORY_PROMPTS:
    print("\n" + "=" * 88)
    print(f"Prompt:  {prompt}")
    print(
        f"Base:    {generate_greedy_continuation(comparison_base_model, prompt, cpt_tokenizer)}"
    )
    print(
        f"Trained: {generate_greedy_continuation(comparison_cpt_model, prompt, cpt_tokenizer)}"
    )

print("\n" + "=" * 88)
print(
    f"Reserved Aria perplexity improvement: {cpt_evidence['heldout_improvement_pct']:+.2f}% "
    f"({cpt_evidence['status']})."
)
print(
    "The perplexity gate is the quantitative evidence. The generations reveal whether that "
    "movement is large enough to create a visibly more story-aligned continuation."
)
if MODEL_PROFILE == "cpu-small-135m":
    print(
        "A fluent or clearly improved continuation is not guaranteed at 135M and ten updates; "
        "unchanged or weak text is an expected capacity-and-budget result."
    )

del comparison_base_model, comparison_cpt_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Weight Movement Layer by Layer

A single aggregate can hide where full fine-tuning changed the network. The cell below samples the same number of individual absolute weight deltas from every transformer block and gives **each block its own panel**.

Figures contain at most 10 panels, so the runtime-reported decoder blocks are paginated automatically. Every panel uses the same y-axis scale: a quiet block therefore cannot look as active as a strongly moving block merely because its axis was automatically rescaled.

> **What to look for:** Compare the mean and maximum in each panel title, then inspect the shape. Long regions near zero mean many sampled weights barely moved; spikes identify sampled weights with larger changes. These are sampled magnitudes, not a claim that one block alone stores the learned behavior.

In [ ]:
# Reload the saved checkpoint and compare its real weight movement with the pinned CPT base.
import gc

base_state = dict(cpt_base_model.named_parameters())
ft_trace_model = AutoModelForCausalLM.from_pretrained(CPT_CHECKPOINT_DIR).to("cpu")

SAMPLES_PER_BLOCK = 500
PANELS_PER_FIGURE = 10
N_COLUMNS = 2

all_deltas = []
for block_index in range(len(cpt_base_model.model.layers)):
    block_deltas = []
    for name, parameter in ft_trace_model.named_parameters():
        if f"model.layers.{block_index}." in name:
            delta = (parameter.data.cpu() - base_state[name].data.cpu()).abs().flatten()
            block_deltas.append(delta)
    if block_deltas:
        combined = torch.cat(block_deltas)
        stride = max(1, len(combined) // SAMPLES_PER_BLOCK)
        all_deltas.append(combined[::stride][:SAMPLES_PER_BLOCK].float().numpy())

global_ymax = max(float(block.max()) for block in all_deltas)
y_limit = global_ymax * 1.05 if global_ymax > 0 else 1e-9

for page_start in range(0, len(all_deltas), PANELS_PER_FIGURE):
    page = all_deltas[page_start : page_start + PANELS_PER_FIGURE]
    row_count = (len(page) + N_COLUMNS - 1) // N_COLUMNS
    figure, axes = plt.subplots(
        row_count,
        N_COLUMNS,
        figsize=(14, 2.6 * row_count),
        sharex=True,
        sharey=True,
        squeeze=False,
    )
    axes = axes.ravel()

    for panel_index, block_values in enumerate(page):
        block_index = page_start + panel_index
        weight_indices = np.arange(len(block_values))
        axis = axes[panel_index]
        axis.plot(weight_indices, block_values, linewidth=0.7, color="steelblue")
        axis.fill_between(weight_indices, block_values, alpha=0.12, color="steelblue")
        axis.set_title(
            f"Transformer block {block_index}  "
            f"(mean={block_values.mean():.2e}, max={block_values.max():.2e})",
            fontsize=9,
        )
        axis.set_ylim(0, y_limit)
        axis.grid(alpha=0.2, axis="y")

    for unused_axis in axes[len(page) :]:
        unused_axis.set_visible(False)

    page_end = page_start + len(page) - 1
    figure.suptitle(
        f"Full Fine-Tuning Weight Movement: Blocks {page_start}-{page_end}",
        fontsize=12,
        fontweight="bold",
    )
    figure.supxlabel(f"Sampled weight index ({SAMPLES_PER_BLOCK} weights per block)")
    figure.supylabel("|W_after - W_before|")
    plt.tight_layout(rect=(0.03, 0.03, 1, 0.96))
    plt.show()

peak_block = max(range(len(all_deltas)), key=lambda index: all_deltas[index].mean())
min_block = min(range(len(all_deltas)), key=lambda index: all_deltas[index].mean())
print(
    f"Mean |delta W| by block - min: block {min_block} "
    f"({all_deltas[min_block].mean():.4e}), max: block {peak_block} "
    f"({all_deltas[peak_block].mean():.4e})."
)

del ft_trace_model
gc.collect()
print("Freed the trace model; the checkpoint and manifest remain on disk.")

### Visualizing Training Progress: Loss Curves

After training completes, inspect the real per-step loss recorded by the `Trainer`, not an idealized illustration. Textbook loss curves are smooth; a 10-step, batch-size-1 CPU demo is usually much noisier, and that is worth seeing honestly.

**What to look for:**

1. **Overall direction:** Compare the first and final logged losses without demanding monotonic progress.
2. **Batch noise:** Each point comes from one paragraph-sized batch, so example difficulty can dominate adjacent steps.
3. **Magnitude:** Lower training loss means a better fit to these batches, not proof of held-out quality.

With all ten steps logged, inspect the complete path rather than one endpoint. A production run would add held-out loss and stop from measured convergence rather than this fixed teaching budget.

> **PyTorch → Keras:** `trainer.state.log_history` — HuggingFace's `Trainer` records a running list of dicts (step number, loss, learning rate, etc.) logged every `logging_steps`; this cell filters that list down to just the `(step, loss)` pairs for plotting. **Keras/TF equivalent:** `history = model.fit(...)` / `history.history["loss"]` — Keras's `fit()` returns a `History` object whose `.history` dict holds per-*epoch* (not per-step, by default) metric lists; matching HuggingFace's per-step granularity in Keras needs a custom callback (e.g. overriding `on_train_batch_end`).

In [ ]:
# Visualize the REAL loss curve from the continued-pretraining run above (trainer_full),
# not a fabricated "typical" curve -- this is exactly what your training just did.
def extract_loss_history(trainer):

    # Pull (step, loss) pairs out of the Trainer's log history, skipping eval-only entries
    return [
        (entry["step"], entry["loss"])
        for entry in trainer.state.log_history
        if "loss" in entry
    ]


full_ft_history = extract_loss_history(trainer_full)
steps, losses = zip(*full_ft_history)  # split into two parallel sequences for plotting

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    steps,
    losses,
    marker="o",
    linewidth=2,
    markersize=7,
    color="green",
    label="Training loss",
)
ax.set_xlabel("Training Step")
ax.set_ylabel("Loss")
ax.set_title(
    f"Continued Pretraining (Full FT): real loss log ({losses[0]:.2f} \u2192 {losses[-1]:.2f})",
    fontsize=12,
    fontweight="bold",
)
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

print(f"\n{'=' * 70}")
print("Reading this REAL loss curve (not an idealized one):")
print(f"{'=' * 70}")
print(f"  Logged steps: {list(steps)}")
print(f"  Logged losses: {[round(l, 3) for l in losses]}")
print(f"  First -> last: {losses[0]:.3f} -> {losses[-1]:.3f}")
print(f"{'=' * 70}")
print("What to look for:")
print(
    "  • A clean, monotonic plateau like a textbook figure is the exception, not the rule --"
)
print(
    "    especially at batch_size=1 with only a handful of steps, loss is dominated by"
)
print("    per-batch noise (which paragraph happened to be in this batch) more than by")
print("    the underlying trend.")
print(
    "  • If the trend is flat/noisy rather than decreasing: raise max_steps, increase the"
)
print(
    "    batch size, or train on more paragraphs so the trend has room to dominate the noise."
)
print(
    "  • Compare this to the loss curves for instruction tuning, partial freezing, and LoRA"
)
print(
    "    continued pretraining further down -- they were all recorded the same real way."
)
print(f"{'=' * 70}")


### Diagnose Continued-Pretraining Failures

| Symptom | Likely cause | First response |
| --- | --- | --- |
| General prompts become nonsense | Too many updates on a narrow corpus | Reduce steps and evaluate domain and general prompts together |
| Training perplexity approaches 1 while held-out perplexity stays high | Memorization | Add diverse text and deduplicate repeated passages |
| Most tokens are padding | `max_length` is much larger than typical paragraphs | Match block length to the observed token-length distribution |
| Training crashes around padding | Causal tokenizer has no pad token | Set `tokenizer.pad_token = tokenizer.eos_token` and mask padding labels with `-100` |

A quick health check needs three probes:

```python
generate(model, "Aria Voss checked the Meridian's Promise and")  # domain fit
generate(model, "The capital of France is")                      # retention
generate(model, "In the Under-Hold, the rebels gathered and")    # generalization
```

A failed general prompt suggests forgetting. A word-for-word held-out continuation suggests memorization. Neither is visible from training loss alone.

## Concept 2: Supervised Fine-Tuning - Specialize the Request/Response Contract

Continued pretraining practices manuscript prose, not an assistant contract. The base checkpoint here is already instruction-tuned, so this section does **not** claim to create general instruction following from scratch. It asks a narrower, attributable question:

> Can SFT make an existing assistant more reliable at Riverside's one-sentence continuation contract?

Riverside supplies demonstrations with two roles:

- **Request:** continue the supplied Aria context in exactly one sentence and stop.
- **Desired response:** the first sentence of the next real manuscript paragraph.

### Instruction Prompts Begin Here

This is the first section that needs role-based prompting. `render_instruction()` uses SmolLM2's native chat template to serialize the system instruction, user request, and assistant response:

```text
<|im_start|>system
You are Riverside House's concise fiction-writing assistant.<|im_end|>
<|im_start|>user
Continue this Aria scene in exactly one sentence and stop.

Context:
...<|im_end|>
<|im_start|>assistant
...<|im_end|>
```

For SFT, the complete serialized sequence is model input, but it is not all a training target. The system and user tokens remain visible as context while their labels become `-100`; only the assistant response, including one EOS token, contributes loss. That response-only supervision is the mechanism that specializes the request/response contract.

Training uses two equivalent request phrasings from the 36 training chapters. Evaluation uses a third, unseen phrasing and contexts from the four reserved chapters. The acceptance metric is complete-contract pass rate under deterministic decoding: exactly one sentence and an EOS stop.

The ten-step CPU run remains deliberately short. A positive held-out pass-rate delta supports the specialization claim; otherwise the notebook reports `INCONCLUSIVE` rather than treating a saved adapter as evidence.

> **Bridge to preference alignment:** SFT can make outputs valid under a contract. Preference data is needed only when multiple valid answers remain and one is more useful than another.

This teaching run uses LoRA to fit local hardware. Part 2 separates that parameter choice from the SFT objective.

> **PyTorch → Keras:** `from peft import LoraConfig, get_peft_model, TaskType` — imports HuggingFace's PEFT library, which wraps a PyTorch model's targeted `nn.Linear` layers with low-rank adapter matrices and freezes everything else; the actual wrapping happens a few cells down. **Keras/TF equivalent:** there is no first-party `peft` support for `TFPreTrainedModel`s — the common Keras/TF pattern for parameter-efficient tuning is manual layer freezing (`layer.trainable = False` on all but the last few layers) rather than LoRA adapters, since PEFT's LoRA implementation is PyTorch-only.

In [ ]:
import gc
import re

from peft import LoraConfig, PeftModel, TaskType, get_peft_model

# The CPT models are no longer needed after their objective-specific evaluation.
for model_name in (
    "trainer_full",
    "full_ft_model",
    "cpt_base_model",
    "base_model",
    "base_state",
):
    if model_name in globals():
        del globals()[model_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Freed continued-pretraining models before LoRA SFT.")

SYSTEM_PROMPT = "You are Riverside House's concise fiction-writing assistant."


def render_instruction(instruction, response=None):
    """Render the native SmolLM2 chat contract used by SFT and later DPO evaluation."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction.strip()},
    ]
    if response is None:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    messages.append({"role": "assistant", "content": response.strip()})
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )


SFT_TRAIN_TASKS = (
    "Continue this Aria scene in exactly one sentence and stop.",
    "Write one sentence that continues this passage, then stop.",
)
SFT_EVAL_TASK = "Advance the supplied scene by one sentence only."
INSTRUCTION_TASK = SFT_TRAIN_TASKS[0]


def first_complete_sentence(text, max_text_tokens=24):
    """Return one complete sentence that fits safely inside the assistant suffix budget."""
    text = text.strip()
    match = re.search(r"^.*?[.!?](?:[\"']?)(?=\s|$)", text)
    sentence = match.group(0).strip() if match else text
    words = sentence.split()

    while words:
        candidate = " ".join(words).rstrip(" ,;:-.!?") + "."
        token_count = len(tokenizer(candidate, add_special_tokens=False)["input_ids"])
        if token_count <= max_text_tokens:
            return candidate
        words.pop()
    raise ValueError("Could not construct a complete bounded response sentence")


def build_instruction_pairs(chapter_files, tasks, max_pairs=None):
    """Build adjacent-paragraph SFT pairs from explicit chapter files and request phrasings."""
    pairs = []
    for path in chapter_files:
        paragraphs = [
            paragraph.strip().replace("\n", " ")
            for paragraph in path.read_text(encoding="utf-8").split("\n\n")
            if len(paragraph.strip()) > 200
        ]
        for pair_index, (context, next_paragraph) in enumerate(
            zip(paragraphs, paragraphs[1:])
        ):
            task = tasks[pair_index % len(tasks)]
            instruction = f"{task}\n\nContext:\n{context}"
            pairs.append(
                {"instruction": instruction, "response": first_complete_sentence(next_paragraph)}
            )
            if max_pairs is not None and len(pairs) >= max_pairs:
                return pairs
    return pairs

### Where These Instruction Pairs Actually Come From

The runnable path constructs examples deterministically from adjacent paragraphs in *The Weight of Distant Light*:

```text
paragraph i     -> instruction context
first sentence of paragraph i + 1 -> desired response
```

Training alternates two equivalent one-sentence requests. Evaluation uses a third phrasing that never appears in training. The text itself is also separated by chapter: training examples come only from the first 36 chapters, while evaluation examples come only from the final four.

| Role | What this notebook uses | Separate model required? |
| --- | --- | --- |
| Pair construction | Python adjacency logic over Aria chapters | No |
| Model being specialized | Pinned SmolLM2 instruction checkpoint | Yes; loaded locally |
| Optional synthetic-pair generator | Not used | No |

This extraction is deliberately narrow. It supports a measurable one-sentence continuation contract and prompt-masking demonstration; it does not establish broad editing, summarization, question-answering, or style-control skill.

### Tokenizing With the Prompt-Mask Pattern

Token-aware budgeting keeps at most 64 native prompt tokens and at most 32 native assistant-suffix tokens inside 96 positions. Prompt and padding labels are `-100`, and exactly one assistant EOS token is supervised.


> **PyTorch → Keras:** `tokenize_instruction()` — builds `labels` as a copy of the tokenized `input_ids`, then overwrites *both* the prompt-token positions and the padding positions with `-100`, so a cross-entropy loss with `ignore_index=-100` (used earlier in the notebook) only ever grades the completion tokens. **Keras/TF equivalent:** the same masking logic — a Keras/TF version would build an analogous `labels` array with `-100` (or `0` plus a matching `sample_weight` mask, since TF's `SparseCategoricalCrossentropy` has no built-in `ignore_index`) at prompt+padding positions; the tokenization itself is identical since `AutoTokenizer` is framework-agnostic.

In [ ]:
_SFT_CONTEXT_MARKER = "\n\nContext:\n"


def _sft_token_ids(text):
    return tokenizer(text, add_special_tokens=False)["input_ids"]


def _sft_prefix_encoding(text):
    if tokenizer.is_fast:
        return tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    return tokenizer(text, add_special_tokens=False)


def _sft_token_prefix_text(text, encoding, token_count):
    if token_count == 0:
        return ""
    offsets = encoding.get("offset_mapping")
    if offsets is not None:
        return text[: offsets[token_count - 1][1]].rstrip()
    return tokenizer.decode(
        encoding["input_ids"][:token_count],
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    ).rstrip()


def _sft_largest_fitting_prefix(text, max_source_tokens, build_candidate):
    encoding = _sft_prefix_encoding(text)
    low = 0
    high = min(len(encoding["input_ids"]), max_source_tokens)
    best = None

    while low <= high:
        token_count = (low + high) // 2
        prefix = _sft_token_prefix_text(text, encoding, token_count)
        candidate = build_candidate(prefix)
        if candidate is None:
            high = token_count - 1
        else:
            best = candidate
            low = token_count + 1

    return best


def _sft_bounded_prompt(instruction, prompt_max_length):
    task, marker, context = instruction.strip().partition(_SFT_CONTEXT_MARKER)
    if not marker:
        raise ValueError("Instruction is missing the expected Context section")

    context = context.strip()
    instruction_prefix = task + marker

    def build_candidate(bounded_context):
        bounded_instruction = instruction_prefix + bounded_context
        prompt_text = render_instruction(bounded_instruction)
        prompt_ids = _sft_token_ids(prompt_text)
        if len(prompt_ids) <= prompt_max_length:
            return bounded_instruction, prompt_text, prompt_ids
        return None

    result = _sft_largest_fitting_prefix(
        context, prompt_max_length, build_candidate
    )
    if result is None:
        raise ValueError("The fixed instruction template exceeds the prompt budget")
    return result


def _sft_bounded_assistant_suffix(
    bounded_instruction,
    prompt_text,
    prompt_ids,
    response,
    assistant_max_length,
):
    if tokenizer.eos_token is None or tokenizer.eos_token_id is None:
        raise ValueError("The tokenizer must define an EOS token")

    def build_candidate(bounded_response):
        full_text = render_instruction(bounded_instruction, bounded_response)
        assert full_text.startswith(prompt_text), (
            "Native chat template did not preserve the exact prompt text prefix"
        )

        assistant_text = full_text[len(prompt_text) :]
        eos_offset = assistant_text.rfind(tokenizer.eos_token)
        if eos_offset < 0:
            raise ValueError("Native assistant rendering did not contain EOS")

        through_eos_text = prompt_text + assistant_text[
            : eos_offset + len(tokenizer.eos_token)
        ]
        through_eos_ids = _sft_token_ids(through_eos_text)
        if through_eos_ids[: len(prompt_ids)] != prompt_ids:
            return None

        assistant_ids = through_eos_ids[len(prompt_ids) :]
        if (
            len(assistant_ids) <= assistant_max_length
            and assistant_ids
            and assistant_ids[-1] == tokenizer.eos_token_id
            and assistant_ids.count(tokenizer.eos_token_id) == 1
        ):
            return assistant_ids
        return None

    result = _sft_largest_fitting_prefix(
        response.strip(), assistant_max_length, build_candidate
    )
    if result is None:
        raise ValueError("No stable native assistant suffix fits the assistant budget")
    return result


def tokenize_instruction(
    example,
    max_length=96,
    prompt_max_length=64,
    assistant_max_length=32,
):
    if prompt_max_length + assistant_max_length != max_length:
        raise ValueError("Prompt and assistant budgets must sum to max_length")
    if tokenizer.pad_token_id is None:
        raise ValueError("The tokenizer must define a padding token")

    bounded_instruction, prompt_text, prompt_ids = _sft_bounded_prompt(
        example["instruction"], prompt_max_length
    )
    assistant_ids = _sft_bounded_assistant_suffix(
        bounded_instruction,
        prompt_text,
        prompt_ids,
        example["response"],
        assistant_max_length,
    )

    active_ids = prompt_ids + assistant_ids
    padding_length = max_length - len(active_ids)
    if padding_length < 0:
        raise AssertionError("Bounded prompt and assistant suffix exceed max_length")

    input_ids = active_ids + [tokenizer.pad_token_id] * padding_length
    attention_mask = [1] * len(active_ids) + [0] * padding_length
    labels = [-100] * len(prompt_ids) + assistant_ids + [-100] * padding_length

    assert assistant_ids[-1] == tokenizer.eos_token_id
    assert assistant_ids.count(tokenizer.eos_token_id) == 1
    assert len(input_ids) == len(attention_mask) == len(labels) == max_length
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


### Building the Instruction Dataset

Build training pairs only from the first 36 Aria chapters. Build the contract suite separately from the four reserved chapters and the unseen evaluation request wording.

In [ ]:
instruction_pairs = build_instruction_pairs(ARIA_TRAIN_FILES, SFT_TRAIN_TASKS)
instruction_holdout_pairs = build_instruction_pairs(
    ARIA_HOLDOUT_FILES,
    (SFT_EVAL_TASK,),
    max_pairs=12,
)
print(
    f"Built {len(instruction_pairs):,} Aria SFT training pairs and "
    f"{len(instruction_holdout_pairs)} reserved contract cases"
)

instruction_dataset = Dataset.from_list(instruction_pairs)
instruction_tokenized = instruction_dataset.map(
    tokenize_instruction,
    remove_columns=["instruction", "response"],
)

for row in instruction_tokenized:
    input_ids = row["input_ids"]
    attention_mask = row["attention_mask"]
    labels = row["labels"]
    assert len(input_ids) == len(attention_mask) == len(labels) == 96
    prompt_boundary = next(index for index, label in enumerate(labels) if label != -100)
    active_length = sum(attention_mask)
    active_response_labels = labels[prompt_boundary:active_length]
    decoded_response = tokenizer.decode(
        active_response_labels[:-1],
        skip_special_tokens=False,
    ).strip()
    assert all(label == -100 or mask == 1 for label, mask in zip(labels, attention_mask))
    assert all(label == -100 for label in labels[:prompt_boundary])
    assert all(label != -100 for label in active_response_labels)
    assert all(label == -100 for label in labels[active_length:])
    assert prompt_boundary <= 64
    assert len(active_response_labels) <= 32
    assert active_response_labels.count(tokenizer.eos_token_id) == 1
    assert active_response_labels[-1] == tokenizer.eos_token_id
    assert decoded_response.endswith(".")

assert set(ARIA_TRAIN_FILES).isdisjoint(ARIA_HOLDOUT_FILES)
assert all(SFT_EVAL_TASK in pair["instruction"] for pair in instruction_holdout_pairs)
print(
    f"PASS: all {len(instruction_tokenized):,} SFT rows preserve the 64/32 boundary "
    "and a complete response sentence; reserved cases remain outside training."
)

### A Quick LoRA Preview for This SFT Run

SFT defines **what behavior is taught**. LoRA only changes **where the update is stored**: `get_peft_model()` freezes the base and adds small trainable correction matrices to selected attention projections.

That is enough detail for this chapter. The next code cell uses LoRA so the SFT run fits local hardware; Part 2 derives the low-rank path, measures its parameter budget, and inspects the real matrices.

> **PyTorch → Keras:** `LoraConfig(...)` and `get_peft_model(...)` target the four SmolLM2 attention
projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and report the resulting trainable fraction.
PEFT's wrapping is PyTorch-specific; a Keras implementation needs a compatible low-rank layer wrapper or
manual custom layers rather than coarse whole-layer freezing.


In [ ]:
# Rank-8 adapters on all four attention projections.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
)

instruct_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION
).to(device)
instruct_lora_model = get_peft_model(instruct_base, lora_config)
instruct_lora_model.print_trainable_parameters()

### Training and Saving the Adapter

Same `Trainer` pattern as continued pretraining, just with the LoRA-wrapped model, the prompt-masked
dataset, and a higher learning rate (`2e-4` vs. `5e-5`) -- LoRA needs a higher LR since it's only
updating a tiny slice of parameters.


> **PyTorch → Keras:** `Trainer(model=instruct_lora_model, ...)` / `trainer_instruct.train()` / `instruct_lora_model.save_pretrained(...)` — the same HuggingFace `Trainer` pattern as the earlier full fine-tuning run, just pointed at the LoRA-wrapped model and the prompt-masked instruction dataset, with a higher learning rate since only the small adapter matrices are being updated. **Keras/TF equivalent:** `model.fit(dataset, epochs=...)` — as with the earlier full fine-tuning cell, a Keras/TF version would call `.fit()` on the (layer-frozen) model instead of `Trainer.train()`; `save_pretrained()` again has an identically-named counterpart on `TFPreTrainedModel`.

In [ ]:
# Short LoRA run; the reserved contract suite below decides whether it demonstrated behavior.
training_args_instruct = TrainingArguments(
    output_dir=str(SFT_CHECKPOINT_DIR),
    per_device_train_batch_size=1,
    max_steps=DEMO_TRAIN_STEPS,
    logging_steps=1,
    save_strategy="no",
    learning_rate=2e-4,
    seed=EXPERIMENT_SEED,
    data_seed=EXPERIMENT_SEED,
    report_to="none",
)

trainer_instruct = Trainer(
    model=instruct_lora_model,
    args=training_args_instruct,
    train_dataset=instruction_tokenized,
)
trainer_instruct.train()
instruct_lora_model.save_pretrained(SFT_CHECKPOINT_DIR)
tokenizer.save_pretrained(SFT_CHECKPOINT_DIR)
write_artifact_manifest(
    stage="supervised-fine-tuning-lora",
    output_dir=SFT_CHECKPOINT_DIR,
    training_files=ARIA_TRAIN_FILES,
    evaluation_files=ARIA_HOLDOUT_FILES,
    training_args=training_args_instruct,
    extra={
        "objective": "response-masked causal language modeling",
        "parameter_strategy": "lora",
        "training_tasks": list(SFT_TRAIN_TASKS),
        "evaluation_task": SFT_EVAL_TASK,
    },
)
print("Saved instruction-tuned LoRA adapter and provenance manifest.")

### Instruction Tuning, Recapped

The preceding cells implement one attributable SFT specialization:

1. `build_instruction_pairs()` turns adjacent Aria paragraphs into one-sentence editor requests and bounded desired responses.
2. The first 36 chapters and two request phrasings supply training; four reserved chapters and unseen wording supply evaluation.
3. `tokenize_instruction()` keeps the request visible but masks its labels with `-100`, so loss grades only the assistant response.
4. A LoRA wrapper keeps the base frozen and stores the update in a small adapter.
5. The reserved complete-contract pass rate decides whether the short run demonstrated behavior.

```text
Continued pretraining: [labels for text .....................] [pad: -100]
Instruction tuning:   [prompt: -100 ........] [completion labels] [pad: -100]
```

The conceptual change is the supervision boundary. The evidentiary change is equally important: correct labels prove the data contract; reserved output checks prove whether behavior moved.

### The Instruction-Tuning Mask Layout, For Real

Contrast this with the continued-pretraining mask layout earlier in the notebook: there, only **padding** was masked, and every real token was active. Here, the **bounded prompt context is masked**, while the **bounded assistant suffix is supervised**. The model is penalized only for the assistant suffix, including exactly one EOS token, never for reproducing the prompt it was given. The cell below takes one real `(prompt, completion)` pair from `instruction_pairs`, runs it through the real `tokenize_instruction()` used for training, and colors every token position by what the label mask actually does with it.


In [ ]:
# Real mask layout for instruction tuning: prompt masked (-100), completion active, padding masked
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

example_pair = instruction_pairs[0]
encoded_example = tokenize_instruction(example_pair)
example_labels = np.array(encoded_example["labels"])
example_attention = np.array(encoded_example["attention_mask"])

# Classify every position: 0 = masked prompt, 1 = active completion, 2 = masked padding
region = np.zeros(len(example_labels), dtype=int)
region[example_attention == 0] = 2  # padding
region[(example_attention == 1) & (example_labels != -100)] = 1  # completion (active)

# everything else (attention==1 & labels==-100) is the masked prompt, stays 0

prompt_masked = int(np.sum(region == 0))
completion_active = int(np.sum(region == 1))
padding_masked = int(np.sum(region == 2))

# Visualize the mask layout as a single color-coded strip (gray=prompt, green=completion, white=padding)
fig, ax = plt.subplots(figsize=(14, 2.2))
cmap = ListedColormap(["lightgray", "mediumseagreen", "white"])
ax.imshow(
    region.reshape(1, -1),
    cmap=cmap,
    aspect="auto",
    vmin=0,
    vmax=2,
    extent=[0, len(region), 0, 1],
)
ax.set_yticks([])
ax.set_xlabel("Token position")
ax.set_title(
    f"{prompt_masked} prompt tokens masked + {completion_active} completion tokens active "
    f"+ {padding_masked} padding tokens masked",
    fontsize=10,
    fontweight="bold",
)

# Legend entries matching each color band in the strip above
legend_handles = [
    Patch(
        facecolor="lightgray", edgecolor="black", label="Prompt (masked, labels=-100)"
    ),
    Patch(
        facecolor="mediumseagreen",
        edgecolor="black",
        label="Completion (active, real labels)",
    ),
    Patch(facecolor="white", edgecolor="black", label="Padding (masked, labels=-100)"),
]
ax.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.55),
    ncol=3,
    fontsize=9,
)

plt.tight_layout()
plt.show()

print(f"Prompt text:     {render_instruction(example_pair['instruction'])[:80]!r}...")
print(f"Completion text: {example_pair['response'][:80]!r}...")
print(
    f"\nMasked (prompt): {prompt_masked} tokens | Active (completion): {completion_active} tokens "
    f"| Masked (padding): {padding_masked} tokens"
)
print(
    "Compare to continued pretraining: there, every real token was active. Here, the prompt is "
    "masked too, so the model only ever gets gradient signal from the completion."
)

### Did SFT Improve the Reserved Contract?

The mask assertions prove that the data pipeline grades only assistant tokens. They do not prove that the adapter changed behavior. The next cell compares the pinned base checkpoint with the SFT adapter on eight cases from reserved chapters using the unseen request wording.

A case passes only when deterministic generation:

1. produces exactly one complete sentence with no trailing fragment; and
2. emits EOS before exhausting the token budget.

The decision is `PASS` when SFT reaches at least 50% complete passes and improves on the base by at least 25 percentage points, `FAIL` when it performs worse than the base, and `INCONCLUSIVE` otherwise. These are transparent teaching thresholds, not production release criteria.

In [ ]:
def generate_contract_result(model, instruction, max_new_tokens=48):
    """Generate deterministically and expose the stopping evidence needed by the contract gate."""
    prompt = render_instruction(instruction)
    encoded = tokenizer(prompt, return_tensors="pt")
    model_device = next(model.parameters()).device
    encoded = {name: tensor.to(model_device) for name, tensor in encoded.items()}
    prompt_length = encoded["input_ids"].shape[1]
    model.eval()
    with torch.no_grad():
        generated = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_ids = generated[0][prompt_length:]
    text = tokenizer.decode(new_ids, skip_special_tokens=True).strip()
    stopped_at_eos = tokenizer.eos_token_id in new_ids.tolist()
    sentence_ends = list(re.finditer(r"[.!?][\"']?(?=\s|$)", text))
    one_complete_sentence = (
        len(sentence_ends) == 1 and not text[sentence_ends[0].end() :].strip()
    )
    return {
        "text": text or "[model stopped immediately]",
        "one_complete_sentence": one_complete_sentence,
        "stopped_at_eos": stopped_at_eos,
        "contract_pass": one_complete_sentence and stopped_at_eos,
    }


def evaluate_contract(model, cases):
    results = [generate_contract_result(model, case["instruction"]) for case in cases]
    return results, sum(result["contract_pass"] for result in results) / len(results)


contract_cases = instruction_holdout_pairs[:8]
contract_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
).to(device)
contract_sft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
).to(device)
contract_sft_model = PeftModel.from_pretrained(
    contract_sft_base,
    SFT_CHECKPOINT_DIR,
).to(device)
try:
    base_contract_results, base_contract_rate = evaluate_contract(contract_base, contract_cases)
    sft_contract_results, sft_contract_rate = evaluate_contract(
        contract_sft_model,
        contract_cases,
    )
finally:
    del contract_base, contract_sft_model, contract_sft_base
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

contract_delta = sft_contract_rate - base_contract_rate
if sft_contract_rate < base_contract_rate:
    sft_status = "FAIL"
elif sft_contract_rate >= 0.50 and contract_delta >= 0.25:
    sft_status = "PASS"
else:
    sft_status = "INCONCLUSIVE"

sft_evidence = {
    "status": sft_status,
    "base_contract_rate": base_contract_rate,
    "sft_contract_rate": sft_contract_rate,
    "delta": contract_delta,
    "cases": len(contract_cases),
}

print("=== SFT held-out contract evidence ===")
print(f"Base complete-pass rate: {base_contract_rate:.1%}")
print(f"SFT complete-pass rate:  {sft_contract_rate:.1%}")
print(f"Difference:              {contract_delta:+.1%}")
print(f"Decision:                {sft_status}")

for case_index, (base_result, sft_result) in enumerate(
    zip(base_contract_results[:4], sft_contract_results[:4]),
    start=1,
):
    print("\n" + "=" * 88)
    print(f"Held-out contract case {case_index}")
    print(
        f"Base [{base_result['contract_pass']}]: "
        f"{base_result['text']}"
    )
    print(
        f"SFT  [{sft_result['contract_pass']}]: "
        f"{sft_result['text']}"
    )

print("\n" + "=" * 88)
print(
    "The pass-rate delta is the contract evidence. The side-by-side text shows whether SFT "
    "also creates a visibly different response rather than only changing EOS behavior."
)

## Concept 3: Failure After SFT - Valid Is Not Yet Preferred

Suppose the SFT adapter now passes Riverside's one-sentence contract. That fixes format compliance, but it exposes a different failure: several outputs can be equally valid while differing sharply in editorial value.

For the same held-out request, imagine two complete one-sentence responses:

```text
Context: Aria watched the prime-number signal repeat across node seventeen.

Response A: Aria isolated the transmission and called Wren to verify that the pattern was artificial.
Response B: Aria considered the signal again and remained where she was while nothing changed.
```

Both responses are grammatical, bounded, and capable of ending at EOS. The contract metric can mark both valid. An editor still prefers A because it advances the scene; B merely restates the setup.

### Why More SFT Does Not Fully Specify the Choice

An SFT example says, "make this demonstrated response likely." It does not directly say how that response should rank against another valid response to the same prompt. Adding more demonstrations may help, but the supervision still hides the comparison Riverside actually cares about.

The missing evidence is relative:

| Field | Meaning |
| --- | --- |
| `prompt` | One shared request and context |
| `chosen` | The response an editor would keep |
| `rejected` | A valid-looking response the editor would reject |

A comparison says only that $y_w$ should outrank $y_l$ for prompt $x$:

$$y_w \succ y_l \mid x.$$

It does not provide an absolute quality score, and it does not imply that suppressing the rejected sentence alone is safe. Probability could move to some third, equally poor response.

Before naming an algorithm, the next training stage therefore needs to:

1. turn many human comparisons into a reusable preference signal;
2. improve responses produced by the current policy, not only memorize fixed demonstrations;
3. limit movement away from the accepted SFT behavior.

The most direct design is to learn an editor-like score, let the model produce fresh drafts, and update the model cautiously from those scores.

### Evolve the Signal: Reward Modeling, Then PPO

A chosen/rejected pair is ordinal evidence, not a reusable score. To judge a fresh draft that never appeared in the comparison dataset, first train a **reward model** $r_\phi(x,y)$ to predict which response a person would prefer.

#### Step 1: Learn the Editor's Ranking Rule

The common Bradley-Terry model turns two reward scores into a preference probability:

$$
P_\phi(y_w \succ y_l \mid x)
= \sigma\!\left(r_\phi(x,y_w)-r_\phi(x,y_l)\right).
$$

The reward-model loss is binary cross-entropy on that difference:

$$
\mathcal{L}_{RM}
= -\mathbb{E}_{(x,y_w,y_l)}
\log \sigma\!\left(r_\phi(x,y_w)-r_\phi(x,y_l)\right).
$$

Only score differences are identifiable: adding the same constant to both rewards changes nothing. The model is learning a ranking surface, not an objective unit of quality.

#### Step 2: Let the Current Policy Produce Fresh Drafts

Now sample $y \sim \pi_{\theta_{old}}(\cdot\mid x)$ from the current policy. This matters because deployed behavior can drift away from the fixed comparison responses. The learned reward model can score those new drafts, but optimizing that score without a constraint invites reward hacking and loss of useful SFT behavior.

Use the accepted SFT model $\pi_{ref}$ as an anchor and form a regularized sequence reward:

$$
R(x,y)
= r_\phi(x,y)
- \beta_{KL}\log\frac{\pi_\theta(y\mid x)}{\pi_{ref}(y\mid x)}.
$$

The second term charges the policy for moving too far from the reference. It is a soft tether, not a guarantee that every capability is preserved.

#### Step 3: Estimate Whether the Draft Was Better Than Expected

A value model $V_\psi(s_t)$ predicts expected future regularized reward from each token state. An advantage estimate asks whether the sampled action did better than that baseline:

$$
\hat A_t \approx \hat R_t - V_\psi(s_t).
$$

Production PPO commonly uses generalized advantage estimation across token steps; the one-number subtraction is the core intuition. Positive advantage means "make this sampled action more likely." Negative advantage means the opposite.

#### Step 4: Update Cautiously With Proximal Policy Optimization

Because the rollout came from the old policy, PPO uses an importance ratio

$$
\rho_t(\theta)
= \frac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{old}}(a_t\mid s_t)}
$$

and maximizes the clipped surrogate

$$
L^{CLIP}(\theta)
= \mathbb{E}_t\!\left[
\min\left(
\rho_t\hat A_t,
\operatorname{clip}(\rho_t,1-\epsilon,1+\epsilon)\hat A_t
\right)
\right].
$$

The clip does not forbid all large updates. It removes the incentive to keep pushing a sampled action beyond the trust region on that batch. PPO still needs learning-rate control, KL monitoring, reward validation, and repeated fresh rollouts.

The full loop is therefore:

1. collect human comparisons;
2. fit the reward model;
3. sample fresh policy responses;
4. score them and estimate advantages with a value model;
5. run several clipped policy/value updates;
6. refresh rollouts and repeat.

This is the natural first answer to the post-SFT failure because it improves behavior the current model actually generates. Its cost is substantial: policy, reference, reward, and value models interact with an on-policy rollout loop. The next code cell isolates PPO's clipping intuition; it is not a claim that this notebook runs RLHF.

In [ ]:
# PPO intuition: see how the clipped surrogate limits incentive on one sampled action.
import math


def ppo_clipped_term(old_logp, new_logp, advantage, clip_epsilon=0.2):
    """Return one token's unclipped and clipped PPO surrogate terms."""
    ratio = math.exp(new_logp - old_logp)
    clipped_ratio = min(max(ratio, 1.0 - clip_epsilon), 1.0 + clip_epsilon)
    raw_term = ratio * advantage
    clipped_term = clipped_ratio * advantage
    objective_term = min(raw_term, clipped_term)
    return {
        "ratio": ratio,
        "clipped_ratio": clipped_ratio,
        "raw_term": raw_term,
        "clipped_term": clipped_term,
        "objective_term": objective_term,
    }


ppo_scenarios = [
    ("helpful, modest increase", 1.10, +0.40),
    ("helpful, aggressive increase", 1.80, +0.40),
    ("harmful, modest decrease", 0.90, -0.30),
    ("harmful, aggressive decrease", 0.50, -0.30),
]

print(f"{'Scenario':31s} {'ratio':>7s} {'A':>7s} {'raw':>8s} {'clipped objective':>18s}")
for label, target_ratio, advantage in ppo_scenarios:
    result = ppo_clipped_term(
        old_logp=0.0,
        new_logp=math.log(target_ratio),
        advantage=advantage,
    )
    print(
        f"{label:31s} {result['ratio']:7.2f} {advantage:+7.2f} "
        f"{result['raw_term']:+8.3f} {result['objective_term']:+18.3f}"
    )

# A sequence-level reward becomes an advantage only after the reference penalty and value baseline.
reward_model_score = 0.38
policy_reference_log_ratio = 0.30
kl_coefficient = 0.10
value_prediction = 0.50
regularized_reward = reward_model_score - kl_coefficient * policy_reference_log_ratio
sequence_advantage = regularized_reward - value_prediction

print()
print("One Riverside rollout:")
print(f"  reward-model score:       {reward_model_score:+.3f}")
print(f"  reference KL charge:      {-kl_coefficient * policy_reference_log_ratio:+.3f}")
print(f"  regularized reward:       {regularized_reward:+.3f}")
print(f"  value prediction:         {value_prediction:+.3f}")
print(f"  resulting advantage:      {sequence_advantage:+.3f}")

assert ppo_clipped_term(0.0, math.log(1.8), +0.4)["objective_term"] == 1.2 * 0.4
assert ppo_clipped_term(0.0, math.log(0.5), -0.3)["objective_term"] == 0.8 * -0.3
print("PASS: clipping limits extra incentive once the policy ratio leaves [0.8, 1.2].")

### From PPO to DPO: Collapse the Offline Machinery

PPO addresses the right failure, but Riverside's teaching run already has a fixed set of comparisons and no environment that requires fresh exploration. A reward model, value model, rollout workers, and on-policy optimization loop may be more machinery than this offline problem needs.

The simplification starts from the KL-regularized policy objective used above. Its optimal policy has the form

$$
\pi^*(y\mid x)
= \frac{1}{Z(x)}\pi_{ref}(y\mid x)
\exp\!\left(\frac{r(x,y)}{\beta}\right).
$$

Rearrange it to express reward through a policy/reference log-ratio:

$$
r(x,y)
= \beta\log\frac{\pi^*(y\mid x)}{\pi_{ref}(y\mid x)}
+ \beta\log Z(x).
$$

For two responses to the same prompt, the unknown partition term $\beta\log Z(x)$ cancels. Substitute the remaining reward difference into the Bradley-Terry preference model and train the policy directly.

Define the chosen-over-rejected implicit reward margin

$$
m_\theta
= \beta\left[
\log\frac{\pi_\theta(y_w\mid x)}{\pi_{ref}(y_w\mid x)}
-
\log\frac{\pi_\theta(y_l\mid x)}{\pi_{ref}(y_l\mid x)}
\right].
$$

**Direct Preference Optimization (DPO)** minimizes

$$
\mathcal{L}_{DPO}
= -\mathbb{E}_{(x,y_w,y_l)}\log\sigma(m_\theta)
= \mathbb{E}\,\operatorname{softplus}(-m_\theta).
$$

This is why DPO belongs after PPO conceptually: it keeps the comparison data and frozen SFT reference, but analytically absorbs the reward model into policy/reference log-ratios and removes fresh rollouts, advantages, the value model, and PPO clipping.

#### Read the Four Log-Probabilities

For each pair, DPO needs:

1. live-policy log-probability of the chosen response;
2. reference-policy log-probability of the chosen response;
3. live-policy log-probability of the rejected response;
4. reference-policy log-probability of the rejected response.

Let

$$
\Delta_w = \log\pi_\theta(y_w\mid x)-\log\pi_{ref}(y_w\mid x),
\qquad
\Delta_l = \log\pi_\theta(y_l\mid x)-\log\pi_{ref}(y_l\mid x).
$$

Then $m_\theta=\beta(\Delta_w-\Delta_l)$. The chosen response may rise, the rejected response may fall, or both may move; DPO cares that the chosen response gains **more relative ground**.

At initialization $\pi_\theta=\pi_{ref}$, so $\Delta_w=\Delta_l=0$, the margin is zero, and the pair loss is $\log 2$. Training should make held-out margins positive.

#### What Was Lost in the Simplification?

DPO is offline. It cannot ask the updated policy for new failure cases during training, and it cannot learn preference coverage absent from the pair dataset. PPO remains the more natural tool when fresh trajectories, environment rewards, tool outcomes, or iterative exploration are essential. DPO is the practical choice here because Riverside has fixed editor comparisons and wants a small, auditable local run.

DPO is not PPO with clipping removed. It is a different offline objective derived from the same KL-regularized preference model.

| Dimension | Reward-model RLHF with PPO | DPO |
| --- | --- | --- |
| Training signal | Scalar reward on fresh rollouts | Chosen/rejected offline pairs |
| Extra learned model | Reward model and value model | None beyond policy; frozen reference still required |
| Data regime | On-policy and iterative | Offline and fixed |
| Stabilizer | PPO ratio clipping plus KL control | Policy/reference log-ratio inside pairwise loss |
| Main strength | Can discover and optimize new policy behavior | Simpler, cheaper, auditable pair training |
| Main blind spot | Reward hacking and unstable multi-model optimization | Pair coverage and distribution shift |

The executable path below now follows the derivation: build chapter-disjoint pairs, compute the four sequence log-probabilities, train against the frozen SFT reference, and test whether the margin transfers to reserved chapters.

In [ ]:
# DPO Step 1: build chapter-disjoint, length-matched preference triples.
DPO_MAX_LENGTH = 128
DPO_MAX_PROMPT_TOKENS = 64
DPO_MAX_RESPONSE_TOKENS = DPO_MAX_LENGTH - DPO_MAX_PROMPT_TOKENS - 1
DPO_TASK = "Continue this Aria scene in exactly one sentence and stop."
STALLING_RESPONSE = (
    "Aria considered the same situation again and repeated what she already knew while "
    "remaining exactly where she was as nothing in the scene changed."
)


def _token_count(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def _bounded_prompt(context):
    """Keep as much Aria context as fits beside the native chat-template overhead."""
    context_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    context_ids = context_ids[:DPO_MAX_PROMPT_TOKENS]
    while context_ids:
        bounded_context = tokenizer.decode(context_ids, skip_special_tokens=True).strip()
        instruction = f"{DPO_TASK}\n\nContext:\n{bounded_context}"
        prompt = render_instruction(instruction)
        if _token_count(prompt) <= DPO_MAX_PROMPT_TOKENS:
            return instruction, prompt
        context_ids = context_ids[:-1]
    raise ValueError("Chat-template overhead leaves no room for DPO context tokens")


def _render_response_suffix(instruction, prompt, response):
    """Render one complete response with exactly one terminal EOS marker."""
    full_text = render_instruction(instruction, response)
    if not full_text.startswith(prompt):
        raise ValueError("Instruction template did not preserve the DPO prompt prefix")
    response_suffix = full_text[len(prompt) :]
    eos_index = response_suffix.find(tokenizer.eos_token)
    if eos_index == -1:
        raise ValueError("Rendered response does not contain the tokenizer EOS marker")
    return response_suffix[: eos_index + len(tokenizer.eos_token)]


def _matched_stalling_suffix(instruction, prompt, target_tokens):
    """Find a complete repetitive sentence with exactly the chosen suffix token count."""
    words = STALLING_RESPONSE.rstrip(".").split()
    for word_count in range(4, len(words) + 1):
        candidate = " ".join(words[:word_count]) + "."
        suffix = _render_response_suffix(instruction, prompt, candidate)
        if _token_count(suffix) == target_tokens:
            return suffix
    return None


def build_preference_pairs(chapter_files, max_pairs):
    """Build real-next-sentence versus repetitive-stall pairs from explicit chapters."""
    pairs = []
    for path in chapter_files:
        paragraphs = [
            paragraph.strip().replace("\n", " ")
            for paragraph in path.read_text(encoding="utf-8").split("\n\n")
            if len(paragraph.strip()) > 200
        ]
        for context, next_paragraph in zip(paragraphs, paragraphs[1:]):
            instruction, prompt = _bounded_prompt(context)
            chosen_text = first_complete_sentence(
                next_paragraph,
                max_text_tokens=DPO_MAX_RESPONSE_TOKENS - 4,
            )
            chosen = _render_response_suffix(instruction, prompt, chosen_text)
            chosen_tokens = _token_count(chosen)
            rejected = _matched_stalling_suffix(
                instruction,
                prompt,
                target_tokens=chosen_tokens,
            )
            if rejected is None:
                continue
            pairs.append({"prompt": prompt, "chosen": chosen, "rejected": rejected})
            if len(pairs) >= max_pairs:
                return pairs
    return pairs


preference_train_pairs = build_preference_pairs(ARIA_TRAIN_FILES, max_pairs=24)
preference_holdout_pairs = build_preference_pairs(ARIA_HOLDOUT_FILES, max_pairs=8)
preference_pairs = preference_train_pairs

if len(preference_train_pairs) < 8 or len(preference_holdout_pairs) < 4:
    raise ValueError("Not enough clean, length-matched DPO pairs were produced")

for split_name, pairs in (
    ("training", preference_train_pairs),
    ("held-out", preference_holdout_pairs),
):
    for pair in pairs:
        prompt_tokens = _token_count(pair["prompt"])
        chosen_tokens = _token_count(pair["chosen"])
        rejected_tokens = _token_count(pair["rejected"])
        assert chosen_tokens == rejected_tokens
        assert prompt_tokens + chosen_tokens <= DPO_MAX_LENGTH
        assert pair["chosen"].removesuffix(tokenizer.eos_token).rstrip().endswith(".")
        assert pair["rejected"].removesuffix(tokenizer.eos_token).rstrip().endswith(".")
    print(f"{split_name.title()} DPO pairs: {len(pairs)}")

example_preference = preference_train_pairs[0]
print("\n=== One length-matched editorial comparison ===")
print(f"Prompt tokens:   {_token_count(example_preference['prompt'])}")
print(f"Chosen tokens:   {_token_count(example_preference['chosen'])}")
print(f"Rejected tokens: {_token_count(example_preference['rejected'])}")
print(f"Chosen:   {example_preference['chosen'][:220]!r}")
print(f"Rejected: {example_preference['rejected'][:220]!r}")

In [ ]:
# Inspect triplets across the full training pool instead of trusting one convenient example.
preview_indices = sorted(
    {
        0,
        len(preference_train_pairs) // 5,
        2 * len(preference_train_pairs) // 5,
        3 * len(preference_train_pairs) // 5,
        4 * len(preference_train_pairs) // 5,
        len(preference_train_pairs) - 1,
    }
)

print(
    f"DPO corpus breadth: {len(preference_train_pairs)} training triplets + "
    f"{len(preference_holdout_pairs)} held-out triplets"
)
for preview_number, pair_index in enumerate(preview_indices, start=1):
    pair = preference_train_pairs[pair_index]
    prompt_context = pair["prompt"].split("Context:\n", maxsplit=1)[-1]
    prompt_context = prompt_context.split(tokenizer.eos_token, maxsplit=1)[0].strip()
    chosen_text = pair["chosen"].removesuffix(tokenizer.eos_token).strip()
    rejected_text = pair["rejected"].removesuffix(tokenizer.eos_token).strip()

    print("\n" + "=" * 88)
    print(f"Representative training triplet {preview_number} (dataset index {pair_index})")
    print(f"Context : {prompt_context[-180:]}")
    print(f"Chosen  : {chosen_text}")
    print(f"Rejected: {rejected_text}")
    print(
        f"Tokens  : chosen={_token_count(pair['chosen'])}, "
        f"rejected={_token_count(pair['rejected'])}"
    )

assert len(preference_train_pairs) >= 24
assert len(preference_holdout_pairs) >= 8
print("\nPASS: the executable DPO demonstration uses a multi-triplet train/holdout corpus.")

### DPO Step 2: Measure the Implicit Preference Margin

The data cells above establish the pair contract. The scorer below now implements the quantities from the derivation rather than relying on trainer-reported loss.

For each complete response, `response_sequence_logprob()` sums next-token log-probabilities only over response tokens:

$$
\log\pi(y\mid x)
= \sum_{t=1}^{T_y}\log\pi(y_t\mid x,y_{<t}).
$$

`measure_preference_snapshot()` computes $\Delta_w$, $\Delta_l$, their edge $\Delta_w-\Delta_l$, and $\operatorname{softplus}[-\beta(\Delta_w-\Delta_l)]$. Exact chosen/rejected token-length matching prevents a longer response from receiving a systematically larger-magnitude sum merely because it contains more terms.

The notebook reports two held-out statistics:

- **mean edge:** whether chosen responses gain more relative ground on average;
- **positive-edge rate:** whether that direction holds across pairs instead of being driven by one large outlier.

Before training, live and reference models are identical, so every edge must be zero. After training, the reserved suite uses chapters that supplied neither SFT nor DPO examples.

> **PyTorch → Keras:** TRL's `DPOTrainer` is PyTorch-only. A Keras implementation would compute these same four sequence log-probabilities and optimize the pairwise softplus loss in a custom `train_step`.

In [ ]:
# DPO Step 2: score the same bounded responses that DPOTrainer optimizes.
def response_sequence_logprob(model, prompt, response):
    """Return one response log-probability sum under the trainer's token contract."""
    training_response = (
        response if response.endswith(tokenizer.eos_token) else response + tokenizer.eos_token
    )
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids
    full_ids = tokenizer(
        prompt + training_response,
        add_special_tokens=False,
        return_tensors="pt",
    ).input_ids

    prompt_length = prompt_ids.shape[1]
    if not torch.equal(full_ids[:, :prompt_length], prompt_ids):
        raise ValueError("Prompt tokens are not a stable prefix of prompt + response")
    if full_ids.shape[1] > DPO_MAX_LENGTH:
        raise ValueError("Diagnostic sequence exceeds the DPO trainer token budget")

    model_device = next(model.parameters()).device
    full_ids = full_ids.to(model_device)
    model.eval()
    with torch.no_grad():
        logits = model(input_ids=full_ids).logits[:, :-1, :].float()
        token_logps = F.log_softmax(logits, dim=-1).gather(
            2,
            full_ids[:, 1:].unsqueeze(-1),
        ).squeeze(-1)

    response_logps = token_logps[:, prompt_length - 1 :]
    return {"sum": response_logps.sum().item(), "tokens": response_logps.shape[1]}


def measure_preference_snapshot(policy, reference, pair, beta):
    """Return chosen/rejected movement, preference edge, and pairwise loss for one pair."""
    policy_chosen = response_sequence_logprob(policy, pair["prompt"], pair["chosen"])
    policy_rejected = response_sequence_logprob(policy, pair["prompt"], pair["rejected"])
    reference_chosen = response_sequence_logprob(
        reference,
        pair["prompt"],
        pair["chosen"],
    )
    reference_rejected = response_sequence_logprob(
        reference,
        pair["prompt"],
        pair["rejected"],
    )

    chosen_movement = policy_chosen["sum"] - reference_chosen["sum"]
    rejected_movement = policy_rejected["sum"] - reference_rejected["sum"]
    edge = chosen_movement - rejected_movement
    loss = F.softplus(torch.tensor(-beta * edge)).item()
    return {
        "chosen_tokens": policy_chosen["tokens"],
        "rejected_tokens": policy_rejected["tokens"],
        "chosen_movement": chosen_movement,
        "rejected_movement": rejected_movement,
        "preference_edge": edge,
        "loss": loss,
    }


def measure_preference_suite(policy, reference, pairs, beta):
    snapshots = [
        measure_preference_snapshot(policy, reference, pair, beta) for pair in pairs
    ]
    mean_edge = sum(item["preference_edge"] for item in snapshots) / len(snapshots)
    positive_edge_rate = sum(
        item["preference_edge"] > 0 for item in snapshots
    ) / len(snapshots)
    mean_loss = sum(item["loss"] for item in snapshots) / len(snapshots)
    return {
        "snapshots": snapshots,
        "mean_edge": mean_edge,
        "positive_edge_rate": positive_edge_rate,
        "mean_loss": mean_loss,
    }


def print_preference_suite(label, suite):
    print(f"\n=== {label} ===")
    print(f"Pairs:              {len(suite['snapshots'])}")
    print(f"Mean edge:          {suite['mean_edge']:+.3f}")
    print(f"Positive-edge rate: {suite['positive_edge_rate']:.1%}")
    print(f"Mean pairwise loss: {suite['mean_loss']:.3f}")


print("Sequence scorer ready for training and held-out preference suites.")

In [ ]:
# DPO Step 3: train on distinct pairs and evaluate movement on reserved Aria chapters.
import gc

from peft import PeftModel
from trl import DPOConfig, DPOTrainer

DPO_BETA = 0.1

# Release completed SFT training objects before loading live and reference DPO copies.
for model_name in (
    "trainer_instruct",
    "instruct_lora_model",
    "instruct_base",
    "dpo_trainer",
    "dpo_policy_model",
    "dpo_policy_base",
    "dpo_reference_model",
    "dpo_reference_base",
):
    if model_name in globals():
        del globals()[model_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Freed completed SFT state before loading the DPO policy and reference.")

dpo_policy_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION
).to(device)
dpo_reference_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION
).to(device)
dpo_policy_model = PeftModel.from_pretrained(
    dpo_policy_base,
    SFT_CHECKPOINT_DIR,
    is_trainable=True,
).to(device)
dpo_reference_model = PeftModel.from_pretrained(
    dpo_reference_base,
    SFT_CHECKPOINT_DIR,
    is_trainable=False,
).to(device)
dpo_reference_model.eval()
for parameter in dpo_reference_model.parameters():
    parameter.requires_grad_(False)

before_dpo_holdout = measure_preference_suite(
    dpo_policy_model,
    dpo_reference_model,
    preference_holdout_pairs,
    DPO_BETA,
)
print_preference_suite(
    "Before DPO: live policy matches frozen SFT on reserved chapters",
    before_dpo_holdout,
)
assert abs(before_dpo_holdout["mean_edge"]) < 1e-4
assert before_dpo_holdout["positive_edge_rate"] == 0.0

dpo_dataset = Dataset.from_list(preference_train_pairs)
dpo_args = DPOConfig(
    output_dir=str(DPO_CHECKPOINT_DIR),
    per_device_train_batch_size=1,
    max_length=DPO_MAX_LENGTH,
    max_steps=DEMO_DPO_STEPS,
    learning_rate=5e-5,
    beta=DPO_BETA,
    logging_steps=1,
    save_strategy="no",
    bf16=False,
    seed=EXPERIMENT_SEED,
    data_seed=EXPERIMENT_SEED,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=dpo_policy_model,
    ref_model=dpo_reference_model,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)
dpo_trainer.train()

after_dpo_training = measure_preference_suite(
    dpo_policy_model,
    dpo_reference_model,
    preference_train_pairs[:4],
    DPO_BETA,
)
after_dpo_holdout = measure_preference_suite(
    dpo_policy_model,
    dpo_reference_model,
    preference_holdout_pairs,
    DPO_BETA,
)
print_preference_suite("After DPO: sampled training pairs", after_dpo_training)
print_preference_suite("After DPO: reserved Aria chapters", after_dpo_holdout)

if (
    after_dpo_holdout["mean_edge"] < 0
    and after_dpo_holdout["positive_edge_rate"] < 0.50
):
    dpo_status = "FAIL"
elif (
    after_dpo_holdout["mean_edge"] > 0
    and after_dpo_holdout["positive_edge_rate"] >= 0.625
):
    dpo_status = "PASS"
else:
    dpo_status = "INCONCLUSIVE"

dpo_evidence = {
    "status": dpo_status,
    "training_mean_edge": after_dpo_training["mean_edge"],
    "heldout_mean_edge": after_dpo_holdout["mean_edge"],
    "heldout_positive_edge_rate": after_dpo_holdout["positive_edge_rate"],
    "heldout_pairs": len(preference_holdout_pairs),
}
print(f"\nDPO held-out decision: {dpo_status}")

dpo_policy_model.save_pretrained(DPO_CHECKPOINT_DIR)
tokenizer.save_pretrained(DPO_CHECKPOINT_DIR)
write_artifact_manifest(
    stage="direct-preference-optimization-lora",
    output_dir=DPO_CHECKPOINT_DIR,
    training_files=ARIA_TRAIN_FILES,
    evaluation_files=ARIA_HOLDOUT_FILES,
    training_args=dpo_args,
    extra={
        "objective": "dpo",
        "rubric": "advance the scene over length-matched repetitive stalling",
        "training_pairs": len(preference_train_pairs),
        "heldout_pairs": len(preference_holdout_pairs),
        "heldout_mean_edge": after_dpo_holdout["mean_edge"],
        "heldout_positive_edge_rate": after_dpo_holdout["positive_edge_rate"],
    },
)

instruct_lora_model = dpo_policy_model
del dpo_trainer, dpo_reference_model, dpo_reference_base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Saved DPO adapter and provenance manifest from the pinned SFT checkpoint.")

In [ ]:
# DPO Step 4: pair held-out margins with matched-seed generations.
from peft import PeftModel


def generate_preference_sample(model, prompt, seed, max_new_tokens=32):
    encoded = tokenizer(prompt, return_tensors="pt")
    model_device = next(model.parameters()).device
    encoded = {name: tensor.to(model_device) for name, tensor in encoded.items()}
    prompt_length = encoded["input_ids"].shape[1]
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    model.eval()
    with torch.no_grad():
        generated = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        generated[0][prompt_length:],
        skip_special_tokens=True,
    ).strip() or "[model stopped immediately]"


comparison_sft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
).to(device)
comparison_sft_model = PeftModel.from_pretrained(
    comparison_sft_base,
    SFT_CHECKPOINT_DIR,
).to(device)

generation_comparisons = []
print("=== Reserved-pair DPO evidence and generated behavior ===")
for pair_index, (pair, snapshot) in enumerate(
    zip(preference_holdout_pairs[:4], after_dpo_holdout["snapshots"][:4]),
    start=1,
):
    seed = EXPERIMENT_SEED + pair_index
    sft_output = generate_preference_sample(
        comparison_sft_model,
        pair["prompt"],
        seed,
    )
    dpo_output = generate_preference_sample(
        dpo_policy_model,
        pair["prompt"],
        seed,
    )
    chosen_text = pair["chosen"].removesuffix(tokenizer.eos_token).strip()
    rejected_text = pair["rejected"].removesuffix(tokenizer.eos_token).strip()
    changed = sft_output != dpo_output
    generation_comparisons.append(
        {"sft": sft_output, "dpo": dpo_output, "changed": changed}
    )

    print("\n" + "=" * 88)
    print(
        f"Held-out triplet {pair_index} | edge={snapshot['preference_edge']:+.3f} | "
        f"generation changed={changed}"
    )
    print(f"Chosen target : {chosen_text}")
    print(f"Rejected target: {rejected_text}")
    print(f"SFT generation: {sft_output}")
    print(f"DPO generation: {dpo_output}")

changed_count = sum(row["changed"] for row in generation_comparisons)
generation_change_rate = changed_count / len(generation_comparisons)
dpo_visible_status = "PASS" if generation_change_rate >= 0.50 else "INCONCLUSIVE"
dpo_evidence["generation_change_rate"] = generation_change_rate
dpo_evidence["visible_status"] = dpo_visible_status
if dpo_visible_status != "PASS":
    dpo_evidence["status"] = "INCONCLUSIVE"

print("\n" + "=" * 88)
print(
    f"Held-out ranking: mean edge={after_dpo_holdout['mean_edge']:+.3f}, "
    f"positive-edge rate={after_dpo_holdout['positive_edge_rate']:.1%}."
)
print(
    f"Visible generation changes: {changed_count}/{len(generation_comparisons)} "
    f"({generation_change_rate:.1%}); decision={dpo_visible_status}."
)
print(
    "Ranking movement is real, but this run is not a visible behavior demonstration unless "
    "matched generations change consistently and qualified review confirms that they improve."
)

del comparison_sft_model, comparison_sft_base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

### Reading the DPO Trace Honestly

Read the two scoreboards separately:

1. The sampled training-pair scoreboard confirms that optimization moved the data it saw.
2. The reserved-chapter scoreboard asks whether the same direction transferred to unseen Aria contexts.
3. Mean preference edge reports average movement toward the chosen response relative to the rejected response.
4. Positive-edge rate prevents one unusually large pair from hiding wrong-direction pairs.

At initialization, policy and reference are identical, so all edges are zero. A useful teaching result has positive mean edge on reserved pairs and moves most reserved pairs in the chosen direction.

### What `beta` controls

Treat `beta` as a reference-strength setting, not a universal quality dial. It scales how strongly policy movement relative to the frozen SFT model enters the pairwise loss. Select it with held-out preference, contract-retention, safety, and diversity evidence.

### What this fixture proves

A positive held-out result shows transfer of one explicit rubric: advance the scene rather than repeat the setup. It does not show that editors broadly prefer generated DPO answers.

A production preference claim still requires:

- held-out prompts with matched SFT and DPO generations;
- blinded, randomized wins, losses, and ties from qualified editors;
- contract, factuality, safety, diversity, and length-balanced slices;
- repeated runs or uncertainty estimates when the decision matters.

### DPO's practical failure modes

- **Coverage ceiling:** offline pairs cannot teach preferences they never contain.
- **Label noise:** inconsistent or weak rubrics create contradictory updates.
- **Length and style shortcuts:** superficial patterns can correlate with `chosen`.
- **Distribution shift:** generated behavior may differ from both responses in the pair dataset.
- **Over-optimization:** preference improvement can trade away factuality or instruction compliance.
- **Reference dependence:** a weak SFT anchor remains a weak starting point.

Riverside's CPU run uses 24 distinct training pairs and eight reserved pairs. That is enough for a falsifiable mechanism-and-transfer demonstration, not a release decision.

---

## Checkpoint Inventory Before the Evidence Ledger

Part 1 writes three artifacts under `checkpoints/llm-finetuning/<profile>/`. Each artifact includes `experiment-manifest.json`, which records the selected model profile, pinned base revision, seed, package versions, training chapter hashes, reserved chapter hashes, and training arguments.

| Artifact within the selected profile | Training signal | Reserved decision |
| --- | --- | --- |
| `non-instruction-full` | Raw Aria next-token prediction | Held-out Aria perplexity, visible knowledge retention, and general-language retention |
| `instruction-lora` | One-sentence prompt/completion demonstrations | Complete-contract pass rate on unseen wording and chapters |
| `preference-dpo` | Length-matched advance-versus-stall comparisons | Mean preference edge and positive-edge rate on reserved pairs |

The profile directory prevents model-size collisions. The artifact directory proves that state was saved; only the reserved metric and visible probes show what that state demonstrated.

---

## Objective-Aligned Evidence Ledger

Do not force checkpoints trained for different objectives through one universal prompt and call the result a ranking. Each stage must answer the question created by its own training signal:

| Objective | Reserved evidence | What counts as success |
| --- | --- | --- |
| Continued pretraining | Token-weighted NLL/perplexity on four untouched Aria chapters | Lower domain perplexity without material general-language regression |
| SFT | Complete one-sentence contract passes on unseen wording and reserved contexts | Higher complete-pass rate than the pinned base |
| DPO | Mean preference edge and positive-edge rate on reserved, length-matched pairs | The labeled preference transfers beyond training contexts |

The next cell prints one ledger from the measured evidence dictionaries. If a training or evaluation stage has not run, it stops and names the missing evidence rather than loading an old artifact silently.

In [ ]:
import pandas as pd
from IPython.display import display

required_evidence = ("cpt_evidence", "sft_evidence", "dpo_evidence")
missing_evidence = [name for name in required_evidence if name not in globals()]
if missing_evidence:
    raise RuntimeError(
        "Run each revised training and reserved-evaluation stage before the evidence ledger: "
        + ", ".join(missing_evidence)
    )

dpo_visible_text = (
    "not measured"
    if "generation_change_rate" not in dpo_evidence
    else f"generation changes {dpo_evidence['generation_change_rate']:.1%}"
)
evidence_rows = [
    {
        "Objective": "Continued pretraining",
        "Reserved measurement": (
            f"Aria PPL {cpt_evidence['heldout_base_ppl']:.2f} -> "
            f"{cpt_evidence['heldout_trained_ppl']:.2f} "
            f"({cpt_evidence['heldout_improvement_pct']:+.2f}% improvement); "
            f"general regression {cpt_evidence['general_regression_pct']:+.2f}%"
        ),
        "Decision": cpt_evidence["status"],
    },
    {
        "Objective": "SFT specialization",
        "Reserved measurement": (
            f"Complete-pass rate {sft_evidence['base_contract_rate']:.1%} -> "
            f"{sft_evidence['sft_contract_rate']:.1%} "
            f"({sft_evidence['delta']:+.1%}; {sft_evidence['cases']} cases)"
        ),
        "Decision": sft_evidence["status"],
    },
    {
        "Objective": "DPO rubric transfer",
        "Reserved measurement": (
            f"Mean edge {dpo_evidence['heldout_mean_edge']:+.3f}; "
            f"positive edge {dpo_evidence['heldout_positive_edge_rate']:.1%} "
            f"({dpo_evidence['heldout_pairs']} pairs); {dpo_visible_text}"
        ),
        "Decision": dpo_evidence["status"],
    },
]

evidence_ledger = pd.DataFrame(evidence_rows)
display(evidence_ledger.style.hide(axis="index"))

stage_decisions = set(evidence_ledger["Decision"])
if "FAIL" in stage_decisions:
    notebook_decision = "FAIL: at least one objective regressed on its reserved gate."
elif stage_decisions == {"PASS"}:
    notebook_decision = "PASS: every objective cleared its reserved metric and visible-output gate."
else:
    notebook_decision = (
        "INCONCLUSIVE: at least one short CPU run needs a different model, data contract, "
        "or measured training budget."
    )

print(notebook_decision)
print(
    "Probability metrics and generated outputs are reported separately; neither is allowed to "
    "stand in for the other."
)

---

## Optional Reference: Resumable Fine-Tuning Job Boundaries

The practical objective story and its reserved evidence are complete above. The remaining cells package continued pretraining, SFT, and DPO as separate resumable jobs with explicit model handoffs and provenance manifests.

This is a job-boundary skeleton, not a production platform. Evaluation gates remain the objective-aligned cells above; immutable release publication, approval records, rollback, serving measurements, and monitoring belong in a dedicated production fine-tuning lifecycle notebook rather than being scattered through concept chapters.

The guarded runner remains disabled. It uses the same 36/4 Aria chapter split and never promotes an artifact automatically.

In [ ]:
from pathlib import Path
import gc

import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint


def _resume_checkpoint(output_path):
    """Return the newest Trainer checkpoint in output_path, if one exists."""
    path = Path(output_path)
    return get_last_checkpoint(str(path)) if path.is_dir() else None


def _release_training_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_continued_pretraining(train_dataset, output_path, max_steps):
    """Run one resumable Aria CPT job from the selected plain base revision."""
    output_path = Path(output_path)
    model = AutoModelForCausalLM.from_pretrained(
        CPT_MODEL_NAME,
        revision=CPT_MODEL_REVISION,
    )
    trainer = None
    try:
        args = TrainingArguments(
            output_dir=str(output_path),
            per_device_train_batch_size=1,
            max_steps=max_steps,
            learning_rate=5e-5,
            logging_steps=max(1, min(10, max_steps)),
            save_strategy="steps",
            save_steps=max(1, min(100, max_steps)),
            save_total_limit=2,
            seed=EXPERIMENT_SEED,
            data_seed=EXPERIMENT_SEED,
            report_to="none",
        )
        trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
        result = trainer.train(resume_from_checkpoint=_resume_checkpoint(output_path))
        trainer.save_model(output_path)
        cpt_tokenizer.save_pretrained(output_path)
        write_artifact_manifest(
            "continued-pretraining-full-job",
            output_path,
            ARIA_TRAIN_FILES,
            ARIA_HOLDOUT_FILES,
            args,
            model_name=CPT_MODEL_NAME,
            model_revision=CPT_MODEL_REVISION,
        )
        return dict(result.metrics)
    finally:
        del trainer, model
        _release_training_memory()

### Resumable SFT Adapter Job

The SFT job writes a LoRA adapter and a provenance manifest while keeping the pinned base revision separate. A later release workflow should verify the manifest and reserved contract evidence before publishing an immutable adapter version.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments


def run_lora_sft(train_dataset, output_path, max_steps):
    """Run one resumable Aria SFT job with a trainable LoRA adapter."""
    output_path = Path(output_path)
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
    )
    model = get_peft_model(
        base_model,
        LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=8,
            lora_alpha=16,
            lora_dropout=0.05,
            target_modules=LORA_TARGET_MODULES,
            bias="none",
        ),
    )
    trainer = None
    try:
        args = TrainingArguments(
            output_dir=str(output_path),
            per_device_train_batch_size=1,
            max_steps=max_steps,
            learning_rate=2e-4,
            logging_steps=max(1, min(10, max_steps)),
            save_strategy="steps",
            save_steps=max(1, min(100, max_steps)),
            save_total_limit=2,
            seed=EXPERIMENT_SEED,
            data_seed=EXPERIMENT_SEED,
            report_to="none",
        )
        trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
        result = trainer.train(resume_from_checkpoint=_resume_checkpoint(output_path))
        model.save_pretrained(output_path)
        tokenizer.save_pretrained(output_path)
        write_artifact_manifest(
            "supervised-fine-tuning-lora-job",
            output_path,
            ARIA_TRAIN_FILES,
            ARIA_HOLDOUT_FILES,
            args,
            extra={"evaluation_task": SFT_EVAL_TASK},
        )
        return dict(result.metrics)
    finally:
        del trainer, model, base_model
        _release_training_memory()

### Resumable DPO Adapter Job

DPO remains a separate auditable job from the accepted SFT adapter. The output manifest records the frozen SFT artifact, `beta`, chapter split, and rubric. Promotion still depends on the reserved preference and SFT-retention gates above.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM
from trl import DPOConfig, DPOTrainer


def run_dpo(train_dataset, sft_adapter_path, output_path, max_steps):
    """Run one resumable Aria DPO job against an explicit frozen SFT reference."""
    sft_adapter_path = Path(sft_adapter_path)
    output_path = Path(output_path)
    policy_base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
    )
    reference_base = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        revision=MODEL_REVISION,
    )
    policy = PeftModel.from_pretrained(
        policy_base,
        sft_adapter_path,
        is_trainable=True,
    )
    reference = PeftModel.from_pretrained(
        reference_base,
        sft_adapter_path,
        is_trainable=False,
    )
    reference.eval()
    for parameter in reference.parameters():
        parameter.requires_grad_(False)

    trainer = None
    try:
        args = DPOConfig(
            output_dir=str(output_path),
            per_device_train_batch_size=1,
            max_length=DPO_MAX_LENGTH,
            max_steps=max_steps,
            learning_rate=5e-5,
            beta=DPO_BETA,
            logging_steps=max(1, min(10, max_steps)),
            save_strategy="steps",
            save_steps=max(1, min(100, max_steps)),
            save_total_limit=2,
            bf16=False,
            seed=EXPERIMENT_SEED,
            data_seed=EXPERIMENT_SEED,
            report_to="none",
        )
        trainer = DPOTrainer(
            model=policy,
            ref_model=reference,
            args=args,
            train_dataset=train_dataset,
            processing_class=tokenizer,
        )
        result = trainer.train(resume_from_checkpoint=_resume_checkpoint(output_path))
        policy.save_pretrained(output_path)
        tokenizer.save_pretrained(output_path)
        write_artifact_manifest(
            "direct-preference-optimization-lora-job",
            output_path,
            ARIA_TRAIN_FILES,
            ARIA_HOLDOUT_FILES,
            args,
            extra={
                "sft_adapter": sft_adapter_path.relative_to(REPO_ROOT).as_posix(),
                "rubric": "advance the scene over length-matched repetitive stalling",
            },
        )
        return dict(result.metrics)
    finally:
        del trainer, policy, reference, policy_base, reference_base
        _release_training_memory()

### Guarded Aria Job Runner

The disabled runner rebuilds each training dataset from the 36 Aria training chapters and runs the three jobs in dependency order. It deliberately does not evaluate, gate, or promote: those responsibilities stay explicit in the reserved-evidence cells and a future production lifecycle notebook.

In [ ]:
import math

RUN_ARIA_JOB_SKELETON = False
ARIA_JOB_EPOCHS = 1

aria_job_metrics = {}
if RUN_ARIA_JOB_SKELETON:
    job_root = PROFILE_CHECKPOINT_DIR / "jobs" / "aria-objectives"
    continued_path = job_root / "continued-pretraining"
    sft_path = job_root / "sft-lora"
    dpo_path = job_root / "dpo"

    job_paragraphs = load_paragraphs(ARIA_TRAIN_FILES)
    job_causal = Dataset.from_dict({"text": job_paragraphs}).map(
        lambda examples: tokenize_causal(examples, cpt_tokenizer),
        batched=True,
        remove_columns=["text"],
    )

    job_instruction_pairs = build_instruction_pairs(
        ARIA_TRAIN_FILES,
        SFT_TRAIN_TASKS,
    )
    job_sft = Dataset.from_list(job_instruction_pairs).map(
        tokenize_instruction,
        remove_columns=["instruction", "response"],
    )

    job_preference_pairs = build_preference_pairs(
        ARIA_TRAIN_FILES,
        max_pairs=24,
    )
    job_dpo = Dataset.from_list(job_preference_pairs)

    print(
        f"Aria job input: {len(ARIA_TRAIN_FILES)} training chapters, "
        f"{len(ARIA_HOLDOUT_FILES)} reserved chapters | "
        f"causal chunks={len(job_causal):,}, SFT pairs={len(job_sft):,}, "
        f"DPO pairs={len(job_dpo):,}"
    )

    continued_steps = ARIA_JOB_EPOCHS * math.ceil(len(job_causal))
    sft_steps = ARIA_JOB_EPOCHS * math.ceil(len(job_sft))
    dpo_steps = ARIA_JOB_EPOCHS * len(job_dpo)

    aria_job_metrics["continued_pretraining"] = run_continued_pretraining(
        train_dataset=job_causal,
        output_path=continued_path,
        max_steps=continued_steps,
    )
    aria_job_metrics["lora_sft"] = run_lora_sft(
        train_dataset=job_sft,
        output_path=sft_path,
        max_steps=sft_steps,
    )
    aria_job_metrics["dpo"] = run_dpo(
        train_dataset=job_dpo,
        sft_adapter_path=sft_path,
        output_path=dpo_path,
        max_steps=dpo_steps,
    )

aria_job_metrics

---

## End of Part 1: The Capability Axis

Each objective now has a CPU-feasible, falsifiable evidence path:

| Objective | Training experience | Reserved measurement | Honest short-run outcome |
| --- | --- | --- | --- |
| Continued pretraining | Raw text from 36 Aria chapters | Token-weighted perplexity on four untouched chapters plus general-language retention | `PASS`, `FAIL`, or `INCONCLUSIVE` |
| SFT | One-sentence demonstrations with prompt masking | Complete-contract pass rate on unseen wording and reserved contexts | `PASS`, `FAIL`, or `INCONCLUSIVE` |
| DPO | 24 length-matched advance-versus-stall pairs | Mean edge and positive-edge rate on eight reserved pairs | `PASS`, `FAIL`, or `INCONCLUSIVE` |

The durable intuition is:

- raw text changes which prose the model expects;
- demonstrations specialize which response contract it follows;
- preference pairs change how it ranks competing responses;
- training loss proves optimization, while reserved objective-aligned metrics prove behavior.

Every checkpoint is paired with a provenance manifest containing the pinned model revision, seed, package versions, chapter hashes, split membership, and training arguments. LoRA remains the practical storage choice for the SFT and DPO CPU runs; Part 2 opens that parameter strategy. Continue to **[Part 2: Parameter-Based Techniques](02-llm-finetuning-parameter-techniques.ipynb)** for update cost, then Part 3 for broader workload and release evidence.